In [1]:
# ============================================================
# Cell 1: Imports
# ============================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             brier_score_loss, confusion_matrix, fbeta_score)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import shap
import joblib
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Seed للتجارب
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("✅ Libraries imported successfully!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

✅ Libraries imported successfully!
PyTorch: 2.5.1+cu121
CUDA Available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Device: cuda


In [7]:
# ============================================================
# Cell 2: Paths & Constants
# ============================================================

# المسارات
DATA_DIR    = "data"
MODELS_DIR  = "models"
RESULTS_DIR = "results"
PLOTS_DIR   = "plots"

TRAIN_PATH = os.path.join(DATA_DIR, "train_forecast24_undersampled005_patientaware.csv")
VAL_PATH   = os.path.join(DATA_DIR, "val_forecast24.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test_forecast24.csv")

# الـ Features المعتمدة (13 feature) — أسماء صحيحة
FEATURE_COLS = [
    'mean_hr', 'mean_spo2', 'sd_hr', 'sd_spo2',
    'skewness_hr', 'skewness_spo2', 'kurtosis_hr', 'kurtosis_spo2',
    'max_xc_hr_spo2', 'min_xc_hr_spo2', 'sub', 'sepsis_window', 'blackout_window'
]

TARGET_COL  = 'y_forecast_24h'
PATIENT_COL = 'new_id'
TIME_COL    = 'seconds_since_birth'

# Hyperparameters
BATCH_SIZE  = 256
EPOCHS      = 50
LR          = 1e-3
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
DROPOUT     = 0.3
SEQ_LEN     = 1

print("✅ Paths & Constants defined!")
print(f"Train : {TRAIN_PATH}")
print(f"Val   : {VAL_PATH}")
print(f"Test  : {TEST_PATH}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

✅ Paths & Constants defined!
Train : data\train_forecast24_undersampled005_patientaware.csv
Val   : data\val_forecast24.csv
Test  : data\test_forecast24.csv
Features (13): ['mean_hr', 'mean_spo2', 'sd_hr', 'sd_spo2', 'skewness_hr', 'skewness_spo2', 'kurtosis_hr', 'kurtosis_spo2', 'max_xc_hr_spo2', 'min_xc_hr_spo2', 'sub', 'sepsis_window', 'blackout_window']


In [8]:
# ============================================================
# Cell 3: Load Data
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("✅ Data loaded successfully!")
print(f"\nTrain shape : {train_df.shape}")
print(f"Val   shape : {val_df.shape}")
print(f"Test  shape : {test_df.shape}")

print(f"\n📊 Train class distribution:")
print(train_df[TARGET_COL].value_counts())
print(f"Positive ratio: {train_df[TARGET_COL].mean():.4f}")

print(f"\n📊 Val class distribution:")
print(val_df[TARGET_COL].value_counts())
print(f"Positive ratio: {val_df[TARGET_COL].mean():.4f}")

print(f"\n📊 Test class distribution:")
print(test_df[TARGET_COL].value_counts())
print(f"Positive ratio: {test_df[TARGET_COL].mean():.4f}")

✅ Data loaded successfully!

Train shape : (45717, 19)
Val   shape : (769979, 18)
Test  shape : (617005, 18)

📊 Train class distribution:
y_forecast_24h
1    30178
0    15539
Name: count, dtype: int64
Positive ratio: 0.6601

📊 Val class distribution:
y_forecast_24h
0    764192
1      5787
Name: count, dtype: int64
Positive ratio: 0.0075

📊 Test class distribution:
y_forecast_24h
0    609323
1      7682
Name: count, dtype: int64
Positive ratio: 0.0125


In [5]:
# تحقق من الأسماء الحقيقية للأعمدة
print("Train columns:")
print(list(train_df.columns))

Train columns:
['new_id', 'seconds_since_birth', 'mean_hr', 'mean_spo2', 'sd_hr', 'sd_spo2', 'skewness_hr', 'skewness_spo2', 'kurtosis_hr', 'kurtosis_spo2', 'max_xc_hr_spo2', 'min_xc_hr_spo2', 'sub', 'sepsis_window', 'blackout_window', 'sex', 'birth_weight', 'y_forecast_24h', 'patient_label']


In [9]:
# ============================================================
# Cell 4: Preprocessing
# ============================================================

# تحقق من missing values
print("🔍 Missing values check:")
print(f"Train: {train_df[FEATURE_COLS].isnull().sum().sum()}")
print(f"Val  : {val_df[FEATURE_COLS].isnull().sum().sum()}")
print(f"Test : {test_df[FEATURE_COLS].isnull().sum().sum()}")

# Features & Labels
X_train = train_df[FEATURE_COLS].values
y_train = train_df[TARGET_COL].values
train_patients = train_df[PATIENT_COL].values
train_times    = train_df[TIME_COL].values

X_val = val_df[FEATURE_COLS].values
y_val = val_df[TARGET_COL].values
val_patients = val_df[PATIENT_COL].values
val_times    = val_df[TIME_COL].values

X_test = test_df[FEATURE_COLS].values
y_test = test_df[TARGET_COL].values
test_patients = test_df[PATIENT_COL].values
test_times    = test_df[TIME_COL].values

# Scaler — نفت على Train فقط، نطبق على الكل
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# حفظ الـ Scaler
joblib.dump(scaler, os.path.join(MODELS_DIR, "lstm_scaler.pkl"))

# Class weight للـ Loss
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(device)

print(f"\n✅ Preprocessing done!")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}   | y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}  | y_test : {y_test.shape}")
print(f"\nScaler saved ✅")
print(f"pos_weight (class balance): {pos_weight.item():.4f}")

🔍 Missing values check:
Train: 0
Val  : 0
Test : 0

✅ Preprocessing done!
X_train: (45717, 13) | y_train: (45717,)
X_val  : (769979, 13)   | y_val  : (769979,)
X_test : (617005, 13)  | y_test : (617005,)

Scaler saved ✅
pos_weight (class balance): 0.5149


In [10]:
# ============================================================
# Cell 5: Dataset & DataLoader
# ============================================================

class SepsisDataset(Dataset):
    def __init__(self, X, y):
        # LSTM يحتاج shape: (batch, seq_len, features)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# إنشاء Datasets
train_dataset = SepsisDataset(X_train, y_train)
val_dataset   = SepsisDataset(X_val,   y_val)
test_dataset  = SepsisDataset(X_test,  y_test)

# إنشاء DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                          shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, 
                          shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, 
                          shuffle=False, num_workers=0, pin_memory=True)

print("✅ Datasets & DataLoaders ready!")
print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")
print(f"\nSample batch shape: {next(iter(train_loader))[0].shape}")

✅ Datasets & DataLoaders ready!
Train batches : 179
Val   batches : 3008
Test  batches : 2411

Sample batch shape: torch.Size([256, 1, 13])


In [11]:
# ============================================================
# Cell 6: LSTM Model Architecture
# ============================================================

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0,
            bidirectional = False
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        logits = self.classifier(last_output)
        return logits.squeeze(1)


# إنشاء النموذج
model = LSTMModel(
    input_size  = len(FEATURE_COLS),
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT
).to(device)

# حساب عدد الـ Parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ LSTM Model created!")
print(f"\n{model}")
print(f"\n📊 Model Complexity:")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Non-trainable        : {total_params - trainable_params:,}")

✅ LSTM Model created!

LSTMModel(
  (lstm): LSTM(13, 128, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

📊 Model Complexity:
Total parameters     : 213,633
Trainable parameters : 213,633
Non-trainable        : 0


In [12]:
# ============================================================
# Cell 7: Optimizer, Loss & Scheduler
# ============================================================

# Loss مع class weight لمعالجة الـ imbalance
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# AdamW أفضل من Adam للتعميم
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = LR,
    weight_decay = 1e-4
)

# Scheduler يخفض LR لما يتوقف التحسن
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode     = 'max',   # نتابع PR-AUC (كلما زاد أحسن)
    factor   = 0.5,     # نص الـ LR
    patience = 5,       # ننتظر 5 epochs
    verbose  = True
)

# Early Stopping
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_score = None
        self.stop       = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            print(f"   EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.stop = True
        else:
            self.best_score = score
            self.counter    = 0

early_stopping = EarlyStopping(patience=10)

print("✅ Optimizer, Loss & Scheduler ready!")
print(f"Optimizer  : AdamW (lr={LR}, weight_decay=1e-4)")
print(f"Loss       : BCEWithLogitsLoss (pos_weight={pos_weight.item():.4f})")
print(f"Scheduler  : ReduceLROnPlateau (patience=5, factor=0.5)")
print(f"Early Stop : patience=10 على PR-AUC")

✅ Optimizer, Loss & Scheduler ready!
Optimizer  : AdamW (lr=0.001, weight_decay=1e-4)
Loss       : BCEWithLogitsLoss (pos_weight=0.5149)
Scheduler  : ReduceLROnPlateau (patience=5, factor=0.5)
Early Stop : patience=10 على PR-AUC


In [13]:
# ============================================================
# Cell 8: Training Loop
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            total_loss += loss.item()

            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y_batch.cpu().numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    avg_loss   = total_loss / len(loader)
    roc_auc    = roc_auc_score(all_labels, all_probs)
    pr_auc     = average_precision_score(all_labels, all_probs)

    return avg_loss, roc_auc, pr_auc, all_probs, all_labels


# ── Training ──
print("🚀 Training started!")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<12}{'ROC-AUC':<12}{'PR-AUC':<12}{'LR'}")
print("-" * 65)

history = {'train_loss': [], 'val_loss': [], 'roc_auc': [], 'pr_auc': []}
best_pr_auc   = 0
best_model_path = os.path.join(MODELS_DIR, "lstm_best.pt")
training_start  = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, roc_auc, pr_auc, _, _ = evaluate(model, val_loader, criterion, device)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(pr_auc)
    early_stopping(pr_auc)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['roc_auc'].append(roc_auc)
    history['pr_auc'].append(pr_auc)

    # حفظ أفضل موديل
    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        torch.save(model.state_dict(), best_model_path)
        flag = " ⭐"
    else:
        flag = ""

    epoch_time = time.time() - epoch_start
    print(f"{epoch:<8}{train_loss:<14.4f}{val_loss:<12.4f}{roc_auc:<12.4f}{pr_auc:<12.4f}{current_lr:.2e}{flag}")

    if early_stopping.stop:
        print(f"\n⏹ Early stopping at epoch {epoch}")
        break

total_time = time.time() - training_start
print(f"\n✅ Training finished!")
print(f"Best PR-AUC : {best_pr_auc:.4f}")
print(f"Total time  : {total_time/60:.2f} minutes")
print(f"Model saved : {best_model_path}")

🚀 Training started!
Epoch   Train Loss    Val Loss    ROC-AUC     PR-AUC      LR
-----------------------------------------------------------------
1       0.3312        0.3903      0.8439      0.5386      1.00e-03 ⭐
2       0.2794        0.4092      0.8464      0.5395      1.00e-03 ⭐
3       0.2763        0.3815      0.8487      0.5420      1.00e-03 ⭐
   EarlyStopping: 1/10
4       0.2748        0.3743      0.8451      0.5390      1.00e-03
   EarlyStopping: 2/10
5       0.2738        0.3744      0.8489      0.5410      1.00e-03
   EarlyStopping: 3/10
6       0.2729        0.3490      0.8466      0.5389      1.00e-03
7       0.2725        0.3423      0.8481      0.5422      1.00e-03 ⭐
   EarlyStopping: 1/10
8       0.2722        0.3483      0.8477      0.5406      1.00e-03
9       0.2714        0.3588      0.8474      0.5429      1.00e-03 ⭐
   EarlyStopping: 1/10
10      0.2710        0.3722      0.8477      0.5407      1.00e-03
   EarlyStopping: 2/10
11      0.2706        0.3521      0

In [14]:
# ============================================================
# Cell 9: Load Best Model
# ============================================================

# تحميل أفضل موديل
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print("✅ Best model loaded!")

# الحصول على predictions للـ Val والـ Test
_, val_roc, val_pr, val_probs, val_labels = evaluate(
    model, val_loader, criterion, device)

_, test_roc, test_pr, test_probs, test_labels = evaluate(
    model, test_loader, criterion, device)

print(f"\n📊 Val  → ROC-AUC: {val_roc:.4f} | PR-AUC: {val_pr:.4f}")
print(f"📊 Test → ROC-AUC: {test_roc:.4f} | PR-AUC: {test_pr:.4f}")

✅ Best model loaded!

📊 Val  → ROC-AUC: 0.8474 | PR-AUC: 0.5429
📊 Test → ROC-AUC: 0.8688 | PR-AUC: 0.5419


In [15]:
# ============================================================
# Cell 10: Threshold Selection using F2-score
# ============================================================

from sklearn.metrics import fbeta_score

# نختار الـ threshold على الـ Val فقط
thresholds = np.arange(0.01, 0.99, 0.01)
best_threshold = 0.5
best_f2        = 0

f2_scores = []
for thresh in thresholds:
    preds = (val_probs >= thresh).astype(int)
    f2    = fbeta_score(val_labels, preds, beta=2, zero_division=0)
    f2_scores.append(f2)
    if f2 > best_f2:
        best_f2        = f2
        best_threshold = thresh

print("✅ Threshold Selection done!")
print(f"Best Threshold : {best_threshold:.2f}")
print(f"Best F2-score  : {best_f2:.4f}")

# حفظ الـ threshold
threshold_info = {
    "best_threshold" : float(best_threshold),
    "best_f2"        : float(best_f2),
    "metric"         : "F2-score",
    "selected_on"    : "validation set"
}
with open(os.path.join(RESULTS_DIR, "lstm_threshold.json"), "w") as f:
    json.dump(threshold_info, f, indent=4)

print(f"Threshold saved ✅")

# رسمة F2 vs Threshold
plt.figure(figsize=(10, 4))
plt.plot(thresholds, f2_scores, color='steelblue', linewidth=2)
plt.axvline(best_threshold, color='red', linestyle='--', 
            label=f'Best={best_threshold:.2f} (F2={best_f2:.4f})')
plt.xlabel('Threshold')
plt.ylabel('F2-score')
plt.title('LSTM — F2-score vs Threshold (Validation Set)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_threshold.png"), dpi=150)
plt.show()
print("Plot saved ✅")

✅ Threshold Selection done!
Best Threshold : 0.80
Best F2-score  : 0.5733
Threshold saved ✅
Plot saved ✅


In [16]:
# ============================================================
# Cell 11: Probability Calibration (Platt Scaling)
# ============================================================

from sklearn.linear_model import LogisticRegression

# Platt Scaling على الـ Val
platt = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
platt.fit(val_probs.reshape(-1, 1), val_labels)

# تطبيق الـ calibration
val_probs_cal  = platt.predict_proba(val_probs.reshape(-1, 1))[:, 1]
test_probs_cal = platt.predict_proba(test_probs.reshape(-1, 1))[:, 1]

# حفظ الـ calibration
cal_params = {
    "coef"      : platt.coef_[0][0],
    "intercept" : platt.intercept_[0],
    "method"    : "Platt Scaling"
}
with open(os.path.join(RESULTS_DIR, "lstm_calibration.json"), "w") as f:
    json.dump(cal_params, f, indent=4)

joblib.dump(platt, os.path.join(MODELS_DIR, "lstm_calibration.pkl"))

# مقارنة قبل وبعد
brier_before = brier_score_loss(val_labels, val_probs)
brier_after  = brier_score_loss(val_labels, val_probs_cal)

print("✅ Calibration done!")
print(f"Brier Score Before : {brier_before:.4f}")
print(f"Brier Score After  : {brier_after:.4f}")

# رسمة Calibration Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, probs, title in zip(
    axes,
    [val_probs, val_probs_cal],
    ["Before Calibration", "After Calibration (Platt)"]):

    fraction_pos, mean_pred = calibration_curve(
        val_labels, probs, n_bins=10, strategy='uniform')

    ax.plot(mean_pred, fraction_pos, 'o-', color='steelblue', label='LSTM')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'LSTM — {title}')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_calibration.png"), dpi=150)
plt.show()
print("Plot saved ✅")

✅ Calibration done!
Brier Score Before : 0.1133
Brier Score After  : 0.0042
Plot saved ✅


In [17]:
# ============================================================
# Cell 12: Window-level Evaluation
# ============================================================

# تطبيق الـ threshold على الـ calibrated probabilities
test_preds_cal = (test_probs_cal >= best_threshold).astype(int)

# حساب كل الـ metrics
acc       = accuracy_score(test_labels, test_preds_cal)
precision = precision_score(test_labels, test_preds_cal, zero_division=0)
recall    = recall_score(test_labels, test_preds_cal, zero_division=0)
f1        = f1_score(test_labels, test_preds_cal, zero_division=0)
f2        = fbeta_score(test_labels, test_preds_cal, beta=2, zero_division=0)
roc_auc   = roc_auc_score(test_labels, test_probs_cal)
pr_auc    = average_precision_score(test_labels, test_probs_cal)
brier     = brier_score_loss(test_labels, test_probs_cal)
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds_cal).ravel()
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
npv         = tn / (tn + fn) if (tn + fn) > 0 else 0

print("=" * 55)
print("   LSTM — Window-level Evaluation (Test Set)")
print("=" * 55)
print(f"  Accuracy    : {acc:.4f}")
print(f"  Precision   : {precision:.4f}")
print(f"  Recall      : {recall:.4f}")
print(f"  Specificity : {specificity:.4f}")
print(f"  F1-score    : {f1:.4f}")
print(f"  F2-score    : {f2:.4f}")
print(f"  PPV         : {ppv:.4f}")
print(f"  NPV         : {npv:.4f}")
print(f"  ROC-AUC     : {roc_auc:.4f}")
print(f"  PR-AUC      : {pr_auc:.4f}")
print(f"  Brier Score : {brier:.4f}")
print("=" * 55)
print(f"\n  Confusion Matrix:")
print(f"  TN={tn:,}  FP={fp:,}")
print(f"  FN={fn:,}   TP={tp:,}")

# حفظ النتائج
window_metrics = {
    "level"       : "window",
    "accuracy"    : acc,
    "precision"   : precision,
    "recall"      : recall,
    "specificity" : specificity,
    "f1"          : f1,
    "f2"          : f2,
    "ppv"         : ppv,
    "npv"         : npv,
    "roc_auc"     : roc_auc,
    "pr_auc"      : pr_auc,
    "brier"       : brier,
    "tp": int(tp), "tn": int(tn),
    "fp": int(fp), "fn": int(fn)
}
with open(os.path.join(RESULTS_DIR, "lstm_window_metrics.json"), "w") as f:
    json.dump(window_metrics, f, indent=4)

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
cm = np.array([[tn, fp], [fn, tp]])
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues',
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['True 0', 'True 1'])
plt.title('LSTM — Confusion Matrix (Window-level)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_confusion_matrix.png"), dpi=150)
plt.show()

print("\n✅ Window-level metrics saved & plotted!")

   LSTM — Window-level Evaluation (Test Set)
  Accuracy    : 0.9875
  Precision   : 0.0000
  Recall      : 0.0000
  Specificity : 1.0000
  F1-score    : 0.0000
  F2-score    : 0.0000
  PPV         : 0.0000
  NPV         : 0.9875
  ROC-AUC     : 0.8688
  PR-AUC      : 0.5419
  Brier Score : 0.0070

  Confusion Matrix:
  TN=609,323  FP=0
  FN=7,682   TP=0

✅ Window-level metrics saved & plotted!


In [18]:
# ============================================================
# Cell 12b: Re-select Threshold على Calibrated Probabilities
# ============================================================

thresholds = np.arange(0.001, 0.5, 0.001)
best_threshold_cal = 0.5
best_f2_cal        = 0

f2_scores_cal = []
for thresh in thresholds:
    preds = (val_probs_cal >= thresh).astype(int)
    f2    = fbeta_score(val_labels, preds, beta=2, zero_division=0)
    f2_scores_cal.append(f2)
    if f2 > best_f2_cal:
        best_f2_cal        = f2
        best_threshold_cal = thresh

print("✅ Threshold re-selected on Calibrated Probabilities!")
print(f"Best Threshold (calibrated) : {best_threshold_cal:.3f}")
print(f"Best F2-score               : {best_f2_cal:.4f}")

# رسمة
plt.figure(figsize=(10, 4))
plt.plot(thresholds, f2_scores_cal, color='steelblue', linewidth=2)
plt.axvline(best_threshold_cal, color='red', linestyle='--',
            label=f'Best={best_threshold_cal:.3f} (F2={best_f2_cal:.4f})')
plt.xlabel('Threshold')
plt.ylabel('F2-score')
plt.title('LSTM — F2-score vs Threshold (Calibrated, Val Set)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_threshold_calibrated.png"), dpi=150)
plt.show()
print("Plot saved ✅")

✅ Threshold re-selected on Calibrated Probabilities!
Best Threshold (calibrated) : 0.169
Best F2-score               : 0.5733
Plot saved ✅


In [19]:
# ============================================================
# Cell 13: Window-level Evaluation (Corrected Threshold)
# ============================================================

# تطبيق الـ threshold الصحيح
test_preds_cal = (test_probs_cal >= best_threshold_cal).astype(int)
val_preds_cal  = (val_probs_cal  >= best_threshold_cal).astype(int)

# حساب كل الـ metrics على الـ Test
acc       = accuracy_score(test_labels, test_preds_cal)
precision = precision_score(test_labels, test_preds_cal, zero_division=0)
recall    = recall_score(test_labels, test_preds_cal, zero_division=0)
f1        = f1_score(test_labels, test_preds_cal, zero_division=0)
f2        = fbeta_score(test_labels, test_preds_cal, beta=2, zero_division=0)
roc_auc   = roc_auc_score(test_labels, test_probs_cal)
pr_auc    = average_precision_score(test_labels, test_probs_cal)
brier     = brier_score_loss(test_labels, test_probs_cal)
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds_cal).ravel()
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
npv         = tn / (tn + fn) if (tn + fn) > 0 else 0

print("=" * 55)
print("   LSTM — Window-level Evaluation (Test Set)")
print("=" * 55)
print(f"  Accuracy    : {acc:.4f}")
print(f"  Precision   : {precision:.4f}")
print(f"  Recall      : {recall:.4f}")
print(f"  Specificity : {specificity:.4f}")
print(f"  F1-score    : {f1:.4f}")
print(f"  F2-score    : {f2:.4f}")
print(f"  PPV         : {ppv:.4f}")
print(f"  NPV         : {npv:.4f}")
print(f"  ROC-AUC     : {roc_auc:.4f}")
print(f"  PR-AUC      : {pr_auc:.4f}")
print(f"  Brier Score : {brier:.4f}")
print("=" * 55)
print(f"\n  Confusion Matrix:")
print(f"  TN={tn:,}  FP={fp:,}")
print(f"  FN={fn:,}   TP={tp:,}")

# حفظ النتائج
window_metrics = {
    "level"       : "window",
    "threshold"   : float(best_threshold_cal),
    "accuracy"    : acc,
    "precision"   : precision,
    "recall"      : recall,
    "specificity" : specificity,
    "f1"          : f1,
    "f2"          : f2,
    "ppv"         : ppv,
    "npv"         : npv,
    "roc_auc"     : roc_auc,
    "pr_auc"      : pr_auc,
    "brier"       : brier,
    "tp": int(tp), "tn": int(tn),
    "fp": int(fp), "fn": int(fn)
}
with open(os.path.join(RESULTS_DIR, "lstm_window_metrics.json"), "w") as f:
    json.dump(window_metrics, f, indent=4)

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
cm = np.array([[tn, fp], [fn, tp]])
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues',
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['True 0', 'True 1'])
plt.title('LSTM — Confusion Matrix (Window-level)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_confusion_matrix.png"), dpi=150)
plt.show()

print("\n✅ Window-level metrics saved & plotted!")

   LSTM — Window-level Evaluation (Test Set)
  Accuracy    : 0.9937
  Precision   : 0.9933
  Recall      : 0.5008
  Specificity : 1.0000
  F1-score    : 0.6659
  F2-score    : 0.5559
  PPV         : 0.9933
  NPV         : 0.9937
  ROC-AUC     : 0.8688
  PR-AUC      : 0.5419
  Brier Score : 0.0070

  Confusion Matrix:
  TN=609,297  FP=26
  FN=3,835   TP=3,847

✅ Window-level metrics saved & plotted!


In [20]:
# ============================================================
# Cell 14: Save Predictions (Patient-wise)
# ============================================================

# Val predictions
val_pred_df = pd.DataFrame({
    'patient_id'   : val_patients,
    'seconds_since_birth' : val_times,
    'y_true'       : val_labels,
    'y_prob'       : val_probs,
    'y_prob_cal'   : val_probs_cal,
    'y_pred'       : val_preds_cal
})

# Test predictions
test_pred_df = pd.DataFrame({
    'patient_id'   : test_patients,
    'seconds_since_birth' : test_times,
    'y_true'       : test_labels,
    'y_prob'       : test_probs,
    'y_prob_cal'   : test_probs_cal,
    'y_pred'       : test_preds_cal
})

# حفظ
val_pred_df.save  = val_pred_df.to_csv(
    os.path.join(RESULTS_DIR, "lstm_val_predictions_patientwise.csv"), index=False)
test_pred_df.to_csv(
    os.path.join(RESULTS_DIR, "lstm_test_predictions_patientwise.csv"), index=False)

print("✅ Predictions saved!")
print(f"Val  predictions : {val_pred_df.shape}")
print(f"Test predictions : {test_pred_df.shape}")
print(f"\nSample:")
print(test_pred_df.head())

✅ Predictions saved!
Val  predictions : (769979, 6)
Test predictions : (617005, 6)

Sample:
   patient_id  seconds_since_birth  y_true    y_prob  y_prob_cal  y_pred
0     1747912               394140     0.0  0.599369    0.024163       0
1     1747912               394740     0.0  0.571786    0.018135       0
2     1747912               395340     0.0  0.547774    0.014107       0
3     1747912               395940     0.0  0.567317    0.017308       0
4     1747912               396540     0.0  0.551867    0.014725       0


In [21]:
# ============================================================
# Cell 15: Patient-level Evaluation
# ============================================================

def patient_level_evaluation(pred_df, dataset_name="Test"):
    results = []

    for patient_id, group in pred_df.groupby('patient_id'):
        y_true_patient = group['y_true'].max()      # 1 لو فيه ولو window واحدة positive
        y_pred_patient = group['y_pred'].max()      # 1 لو النموذج حذّر ولو مرة
        y_prob_patient = group['y_prob_cal'].max()  # أعلى احتمال للمريض

        results.append({
            'patient_id'    : patient_id,
            'y_true_patient': y_true_patient,
            'y_pred_patient': y_pred_patient,
            'y_prob_patient': y_prob_patient
        })

    results_df = pd.DataFrame(results)

    y_true_p = results_df['y_true_patient'].values
    y_pred_p = results_df['y_pred_patient'].values
    y_prob_p = results_df['y_prob_patient'].values

    tn, fp, fn, tp = confusion_matrix(y_true_p, y_pred_p).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv         = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1          = f1_score(y_true_p, y_pred_p, zero_division=0)
    f2          = fbeta_score(y_true_p, y_pred_p, beta=2, zero_division=0)

    print(f"=" * 55)
    print(f"   LSTM — Patient-level Evaluation ({dataset_name})")
    print(f"=" * 55)
    print(f"  Total Patients   : {len(results_df):,}")
    print(f"  Sepsis Patients  : {int(y_true_p.sum()):,}")
    print(f"  Normal Patients  : {int((y_true_p==0).sum()):,}")
    print(f"-" * 55)
    print(f"  Sensitivity      : {sensitivity:.4f}")
    print(f"  Specificity      : {specificity:.4f}")
    print(f"  PPV              : {ppv:.4f}")
    print(f"  NPV              : {npv:.4f}")
    print(f"  F1-score         : {f1:.4f}")
    print(f"  F2-score         : {f2:.4f}")
    print(f"-" * 55)
    print(f"  Confusion Matrix:")
    print(f"  TN={tn:,}  FP={fp:,}")
    print(f"  FN={fn:,}    TP={tp:,}")
    print(f"=" * 55)

    # حفظ النتائج
    patient_metrics = {
        "level"        : "patient",
        "dataset"      : dataset_name,
        "total"        : len(results_df),
        "sepsis"       : int(y_true_p.sum()),
        "normal"       : int((y_true_p == 0).sum()),
        "sensitivity"  : sensitivity,
        "specificity"  : specificity,
        "ppv"          : ppv,
        "npv"          : npv,
        "f1"           : f1,
        "f2"           : f2,
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn)
    }
    with open(os.path.join(RESULTS_DIR, f"lstm_patient_metrics_{dataset_name.lower()}.json"), "w") as f:
        json.dump(patient_metrics, f, indent=4)

    return results_df

# تقييم على الـ Test
test_patient_df = patient_level_evaluation(test_pred_df, "Test")

print("\n✅ Patient-level evaluation done & saved!")

   LSTM — Patient-level Evaluation (Test)
  Total Patients   : 66
  Sepsis Patients  : 22
  Normal Patients  : 44
-------------------------------------------------------
  Sensitivity      : 1.0000
  Specificity      : 1.0000
  PPV              : 1.0000
  NPV              : 1.0000
  F1-score         : 1.0000
  F2-score         : 1.0000
-------------------------------------------------------
  Confusion Matrix:
  TN=44  FP=0
  FN=0    TP=22

✅ Patient-level evaluation done & saved!


In [22]:
print("Test patients count:", test_pred_df['patient_id'].nunique())
print("Val patients count:", val_pred_df['patient_id'].nunique())
print("\nTest sepsis patients:", 
      test_pred_df.groupby('patient_id')['y_true'].max().sum())
print("Test normal patients:", 
      (test_pred_df.groupby('patient_id')['y_true'].max() == 0).sum())

Test patients count: 66
Val patients count: 66

Test sepsis patients: 22.0
Test normal patients: 44


In [23]:
# نشوف توزيع الـ windows لكل مريض
windows_per_patient = test_pred_df.groupby('patient_id').size()
print("Windows per patient:")
print(windows_per_patient.describe())

# نشوف كم window positive لكل مريض sepsis
sepsis_patients = test_pred_df.groupby('patient_id')['y_true'].max()
sepsis_ids = sepsis_patients[sepsis_patients == 1].index

print(f"\nSepsis patients details:")
for pid in sepsis_ids:
    group = test_pred_df[test_pred_df['patient_id'] == pid]
    total_windows    = len(group)
    positive_windows = group['y_true'].sum()
    predicted_pos    = group['y_pred'].sum()
    print(f"  Patient {pid}: {total_windows} windows | "
          f"true_pos={int(positive_windows)} | predicted={int(predicted_pos)}")

Windows per patient:
count       66.000000
mean      9348.560606
std       7326.672564
min         19.000000
25%       3483.750000
50%       8104.500000
75%      12919.500000
max      29525.000000
dtype: float64

Sepsis patients details:
  Patient 1747912: 22055 windows | true_pos=574 | predicted=288
  Patient 3759861: 11230 windows | true_pos=287 | predicted=144
  Patient 10388384: 8230 windows | true_pos=287 | predicted=144
  Patient 12436844: 7142 windows | true_pos=287 | predicted=144
  Patient 15910711: 25575 windows | true_pos=570 | predicted=285
  Patient 19468305: 17240 windows | true_pos=287 | predicted=144
  Patient 22622897: 13352 windows | true_pos=573 | predicted=287
  Patient 24652567: 23015 windows | true_pos=282 | predicted=144
  Patient 28485640: 16375 windows | true_pos=287 | predicted=144
  Patient 33625882: 4233 windows | true_pos=286 | predicted=143
  Patient 38420775: 17264 windows | true_pos=568 | predicted=282
  Patient 64363766: 27756 windows | true_pos=574 | p

In [24]:
# ============================================================
# Cell 16: Alert-based Patient Evaluation (الصحيح طبياً)
# ============================================================

def alert_based_evaluation(pred_df, dataset_name="Test"):
    patient_results = []

    for patient_id, group in pred_df.groupby('patient_id'):
        group = group.sort_values('seconds_since_birth')

        y_true_patient = group['y_true'].max()

        # أول وقت فعلي للـ sepsis
        sepsis_windows = group[group['y_true'] == 1]
        t_event = sepsis_windows['seconds_since_birth'].min() \
                  if len(sepsis_windows) > 0 else None

        # أول alert من النموذج
        alert_windows = group[group['y_pred'] == 1]
        t_alert = alert_windows['seconds_since_birth'].min() \
                  if len(alert_windows) > 0 else None

        # lead time
        if t_event is not None and t_alert is not None:
            lead_seconds = t_event - t_alert
            lead_hours   = lead_seconds / 3600
            detected     = 1 if t_alert <= t_event else 0
        elif t_event is None and t_alert is None:
            lead_hours = None
            detected   = 0  # normal, no alert = correct
        elif t_event is None and t_alert is not None:
            lead_hours = None
            detected   = 0  # false alarm
        else:
            lead_hours = None
            detected   = 0  # missed

        patient_results.append({
            'patient_id'     : patient_id,
            'y_true_patient' : int(y_true_patient),
            'y_pred_patient' : 1 if t_alert is not None else 0,
            't_event_sec'    : t_event,
            't_alert_sec'    : t_alert,
            'lead_hours'     : lead_hours,
            'detected'       : detected
        })

    results_df = pd.DataFrame(patient_results)

    # metrics
    y_true_p = results_df['y_true_patient'].values
    y_pred_p = results_df['y_pred_patient'].values

    tn, fp, fn, tp = confusion_matrix(y_true_p, y_pred_p).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv         = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1          = f1_score(y_true_p, y_pred_p, zero_division=0)
    f2          = fbeta_score(y_true_p, y_pred_p, beta=2, zero_division=0)

    # lead time على المرضى المكتشفين فقط
    detected_df  = results_df[results_df['detected'] == 1]
    mean_lead    = detected_df['lead_hours'].mean() if len(detected_df) > 0 else 0
    median_lead  = detected_df['lead_hours'].median() if len(detected_df) > 0 else 0

    print(f"=" * 57)
    print(f"   LSTM — Alert-based Patient Evaluation ({dataset_name})")
    print(f"=" * 57)
    print(f"  Total Patients    : {len(results_df):,}")
    print(f"  Sepsis Patients   : {int(y_true_p.sum()):,}")
    print(f"  Normal Patients   : {int((y_true_p==0).sum()):,}")
    print(f"-" * 57)
    print(f"  Sensitivity       : {sensitivity:.4f}")
    print(f"  Specificity       : {specificity:.4f}")
    print(f"  PPV               : {ppv:.4f}")
    print(f"  NPV               : {npv:.4f}")
    print(f"  F1-score          : {f1:.4f}")
    print(f"  F2-score          : {f2:.4f}")
    print(f"-" * 57)
    print(f"  Confusion Matrix:")
    print(f"  TN={tn}  FP={fp}")
    print(f"  FN={fn}   TP={tp}")
    print(f"-" * 57)
    print(f"  Detected Sepsis   : {len(detected_df)}/{int(y_true_p.sum())}")
    print(f"  Mean  Lead Time   : {mean_lead:.2f} hours")
    print(f"  Median Lead Time  : {median_lead:.2f} hours")
    print(f"=" * 57)

    # حفظ
    alert_metrics = {
        "level"          : "patient_alert",
        "dataset"        : dataset_name,
        "total_patients" : len(results_df),
        "sepsis"         : int(y_true_p.sum()),
        "normal"         : int((y_true_p == 0).sum()),
        "sensitivity"    : sensitivity,
        "specificity"    : specificity,
        "ppv"            : ppv,
        "npv"            : npv,
        "f1"             : f1,
        "f2"             : f2,
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn),
        "detected_sepsis"   : len(detected_df),
        "mean_lead_hours"   : mean_lead,
        "median_lead_hours" : median_lead
    }
    with open(os.path.join(RESULTS_DIR,
              f"lstm_alert_metrics_{dataset_name.lower()}.json"), "w") as f:
        json.dump(alert_metrics, f, indent=4)

    # حفظ ملف المرضى
    results_df.to_csv(os.path.join(RESULTS_DIR,
                      f"lstm_patient_first_alerts_{dataset_name.lower()}.csv"),
                      index=False)

    return results_df

# تشغيل على Test
test_alert_df = alert_based_evaluation(test_pred_df, "Test")
print("\n✅ Alert-based evaluation done & saved!")

   LSTM — Alert-based Patient Evaluation (Test)
  Total Patients    : 66
  Sepsis Patients   : 22
  Normal Patients   : 44
---------------------------------------------------------
  Sensitivity       : 1.0000
  Specificity       : 1.0000
  PPV               : 1.0000
  NPV               : 1.0000
  F1-score          : 1.0000
  F2-score          : 1.0000
---------------------------------------------------------
  Confusion Matrix:
  TN=44  FP=0
  FN=0   TP=22
---------------------------------------------------------
  Detected Sepsis   : 0/22
  Mean  Lead Time   : 0.00 hours
  Median Lead Time  : 0.00 hours

✅ Alert-based evaluation done & saved!


In [25]:
# ============================================================
# Cell 16b: Debug — نفهم شو صاير
# ============================================================

# نشوف مريض sepsis واحد بالتفصيل
sample_sepsis_id = test_pred_df[
    test_pred_df['patient_id'].isin(
        test_pred_df.groupby('patient_id')['y_true'].max()[
            lambda x: x == 1].index)]['patient_id'].iloc[0]

sample = test_pred_df[test_pred_df['patient_id'] == sample_sepsis_id].sort_values('seconds_since_birth')

print(f"Patient ID: {sample_sepsis_id}")
print(f"Total windows : {len(sample)}")
print(f"y_true values : {sample['y_true'].unique()}")
print(f"y_pred values : {sample['y_pred'].unique()}")
print(f"y_true sum    : {sample['y_true'].sum()}")
print(f"y_pred sum    : {sample['y_pred'].sum()}")
print(f"\nFirst 10 rows:")
print(sample[['seconds_since_birth','y_true','y_pred','y_prob_cal']].head(10))
print(f"\nLast 10 rows:")
print(sample[['seconds_since_birth','y_true','y_pred','y_prob_cal']].tail(10))

Patient ID: 1747912
Total windows : 22055
y_true values : [0. 1.]
y_pred values : [0 1]
y_true sum    : 574.0
y_pred sum    : 288

First 10 rows:
   seconds_since_birth  y_true  y_pred  y_prob_cal
0               394140     0.0       0    0.024163
1               394740     0.0       0    0.018135
2               395340     0.0       0    0.014107
3               395940     0.0       0    0.017308
4               396540     0.0       0    0.014725
5               397140     0.0       0    0.012267
6               397740     0.0       0    0.018142
7               398340     0.0       0    0.007795
8               398940     0.0       0    0.004700
9               399540     0.0       0    0.000971

Last 10 rows:
       seconds_since_birth  y_true  y_pred  y_prob_cal
22045             13904340     0.0       0    0.000067
22046             13904940     0.0       0    0.000058
22047             13905540     0.0       0    0.000048
22048             13906140     0.0       0    0.000063
220

In [26]:
# ============================================================
# Cell 16c: نشوف وين الـ positive windows
# ============================================================

sample = test_pred_df[
    test_pred_df['patient_id'] == sample_sepsis_id
].sort_values('seconds_since_birth').reset_index(drop=True)

# وين الـ y_true = 1
positive_idx = sample[sample['y_true'] == 1.0]
print(f"Positive windows range:")
print(f"  First positive at index : {positive_idx.index[0]}")
print(f"  Last  positive at index : {positive_idx.index[-1]}")
print(f"  seconds_since_birth min : {positive_idx['seconds_since_birth'].min()}")
print(f"  seconds_since_birth max : {positive_idx['seconds_since_birth'].max()}")

# وين الـ y_pred = 1
predicted_idx = sample[sample['y_pred'] == 1]
print(f"\nPredicted positive windows:")
print(f"  Count : {len(predicted_idx)}")
if len(predicted_idx) > 0:
    print(f"  First alert at seconds : {predicted_idx['seconds_since_birth'].min()}")
    print(f"  Last  alert at seconds : {predicted_idx['seconds_since_birth'].max()}")

# نشوف المنطقة حول الـ positive windows
print(f"\nAround positive windows:")
print(sample[['seconds_since_birth','y_true','y_pred','y_prob_cal']].iloc[
    max(0, positive_idx.index[0]-3) : positive_idx.index[0]+10])

Positive windows range:
  First positive at index : 294
  Last  positive at index : 3200
  seconds_since_birth min : 570540
  seconds_since_birth max : 2323740

Predicted positive windows:
  Count : 288
  First alert at seconds : 656940
  Last  alert at seconds : 2324340

Around positive windows:
     seconds_since_birth  y_true  y_pred  y_prob_cal
291               568740     0.0       0    0.022786
292               569340     0.0       0    0.017871
293               569940     0.0       0    0.011989
294               570540     1.0       0    0.017530
295               571140     1.0       0    0.020550
296               571740     1.0       0    0.016707
297               572340     1.0       0    0.021880
298               572940     1.0       0    0.037187
299               573540     1.0       0    0.012893
300               574140     1.0       0    0.010295
301               574740     1.0       0    0.029348
302               575340     1.0       0    0.016810
303          

In [27]:
# ============================================================
# Cell 17: Alert-based Evaluation (النسخة الصحيحة)
# ============================================================

def alert_based_evaluation_v2(pred_df, dataset_name="Test"):
    patient_results = []

    for patient_id, group in pred_df.groupby('patient_id'):
        group = group.sort_values('seconds_since_birth')

        y_true_patient = group['y_true'].max()

        # أول وقت فعلي للـ sepsis
        sepsis_windows = group[group['y_true'] == 1]
        t_event = sepsis_windows['seconds_since_birth'].min() \
                  if len(sepsis_windows) > 0 else None

        # أول alert من النموذج
        alert_windows = group[group['y_pred'] == 1]
        t_alert = alert_windows['seconds_since_birth'].min() \
                  if len(alert_windows) > 0 else None

        # lead time (양수 = مبكر، سالب = متأخر)
        if t_event is not None and t_alert is not None:
            lead_seconds = t_event - t_alert
            lead_hours   = lead_seconds / 3600
            # مكتشف = حذّر قبل الحدث أو بنفس الوقت
            detected = 1 if t_alert <= t_event else 0
        elif t_event is None and t_alert is None:
            lead_hours = None
            detected   = -1  # normal, no alert
        elif t_event is None and t_alert is not None:
            lead_hours = None
            detected   = -2  # false alarm
        else:
            lead_hours = None
            detected   = 0   # missed

        patient_results.append({
            'patient_id'     : patient_id,
            'y_true_patient' : int(y_true_patient),
            'y_pred_patient' : 1 if t_alert is not None else 0,
            't_event_sec'    : t_event,
            't_alert_sec'    : t_alert,
            'lead_hours'     : lead_hours,
            'detected'       : detected,
            'early_warning'  : 1 if detected == 1 else 0
        })

    results_df = pd.DataFrame(patient_results)

    # metrics
    y_true_p = results_df['y_true_patient'].values
    y_pred_p = results_df['y_pred_patient'].values

    tn, fp, fn, tp = confusion_matrix(y_true_p, y_pred_p).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv         = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1          = f1_score(y_true_p, y_pred_p, zero_division=0)
    f2          = fbeta_score(y_true_p, y_pred_p, beta=2, zero_division=0)

    # تحليل الـ sepsis patients
    sepsis_df   = results_df[results_df['y_true_patient'] == 1]
    early_count = sepsis_df['early_warning'].sum()
    late_count  = len(sepsis_df) - early_count - sepsis_df[sepsis_df['detected']==0].shape[0]
    missed      = (sepsis_df['detected'] == 0).sum()

    # lead time على المبكرين فقط
    early_df    = sepsis_df[sepsis_df['early_warning'] == 1]
    mean_lead   = early_df['lead_hours'].mean()   if len(early_df) > 0 else 0
    median_lead = early_df['lead_hours'].median() if len(early_df) > 0 else 0

    print(f"=" * 57)
    print(f"  LSTM — Alert-based Evaluation ({dataset_name})")
    print(f"=" * 57)
    print(f"  Total Patients     : {len(results_df):,}")
    print(f"  Sepsis Patients    : {int(y_true_p.sum()):,}")
    print(f"  Normal Patients    : {int((y_true_p==0).sum()):,}")
    print(f"-" * 57)
    print(f"  Sensitivity        : {sensitivity:.4f}")
    print(f"  Specificity        : {specificity:.4f}")
    print(f"  PPV                : {ppv:.4f}")
    print(f"  NPV                : {npv:.4f}")
    print(f"  F1-score           : {f1:.4f}")
    print(f"  F2-score           : {f2:.4f}")
    print(f"-" * 57)
    print(f"  Confusion Matrix:")
    print(f"  TN={tn}  FP={fp}")
    print(f"  FN={fn}   TP={tp}")
    print(f"-" * 57)
    print(f"  🟢 Early Warning   : {early_count}/{int(y_true_p.sum())} patients")
    print(f"  🔴 Late Warning    : {int(late_count)}/{int(y_true_p.sum())} patients")
    print(f"  ⚫ Missed          : {int(missed)}/{int(y_true_p.sum())} patients")
    print(f"  Mean  Lead Time    : {mean_lead:.2f} hours")
    print(f"  Median Lead Time   : {median_lead:.2f} hours")
    print(f"=" * 57)

    # حفظ
    alert_metrics = {
        "level"             : "patient_alert",
        "dataset"           : dataset_name,
        "total_patients"    : len(results_df),
        "sepsis"            : int(y_true_p.sum()),
        "normal"            : int((y_true_p == 0).sum()),
        "sensitivity"       : sensitivity,
        "specificity"       : specificity,
        "ppv"               : ppv,
        "npv"               : npv,
        "f1"                : f1,
        "f2"                : f2,
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn),
        "early_warning"     : int(early_count),
        "late_warning"      : int(late_count),
        "missed"            : int(missed),
        "mean_lead_hours"   : float(mean_lead),
        "median_lead_hours" : float(median_lead)
    }
    with open(os.path.join(RESULTS_DIR,
              f"lstm_alert_metrics_{dataset_name.lower()}.json"), "w") as f:
        json.dump(alert_metrics, f, indent=4)

    results_df.to_csv(os.path.join(RESULTS_DIR,
                      f"lstm_patient_first_alerts_{dataset_name.lower()}.csv"),
                      index=False)

    return results_df

# تشغيل
test_alert_df = alert_based_evaluation_v2(test_pred_df, "Test")
print("\n✅ Alert-based evaluation done & saved!")

  LSTM — Alert-based Evaluation (Test)
  Total Patients     : 66
  Sepsis Patients    : 22
  Normal Patients    : 44
---------------------------------------------------------
  Sensitivity        : 1.0000
  Specificity        : 1.0000
  PPV                : 1.0000
  NPV                : 1.0000
  F1-score           : 1.0000
  F2-score           : 1.0000
---------------------------------------------------------
  Confusion Matrix:
  TN=44  FP=0
  FN=0   TP=22
---------------------------------------------------------
  🟢 Early Warning   : 0/22 patients
  🔴 Late Warning    : 0/22 patients
  ⚫ Missed          : 22/22 patients
  Mean  Lead Time    : 0.00 hours
  Median Lead Time   : 0.00 hours

✅ Alert-based evaluation done & saved!


In [28]:
# ============================================================
# Cell 17b: Debug التناقض
# ============================================================

sample_id = test_pred_df[
    test_pred_df['patient_id'].isin(
        test_pred_df.groupby('patient_id')['y_true']
        .max()[lambda x: x == 1].index)
]['patient_id'].iloc[0]

group = test_pred_df[
    test_pred_df['patient_id'] == sample_id
].sort_values('seconds_since_birth')

t_event = group[group['y_true'] == 1]['seconds_since_birth'].min()
t_alert = group[group['y_pred'] == 1]['seconds_since_birth'].min() \
          if len(group[group['y_pred'] == 1]) > 0 else None

print(f"Patient      : {sample_id}")
print(f"t_event      : {t_event}")
print(f"t_alert      : {t_alert}")
print(f"t_alert type : {type(t_alert)}")
print(f"Comparison   : t_alert <= t_event → {t_alert <= t_event if t_alert else 'N/A'}")
print(f"lead_hours   : {(t_event - t_alert)/3600 if t_alert else 'N/A'}")
print(f"\ny_pred unique: {group['y_pred'].unique()}")
print(f"y_pred sum   : {group['y_pred'].sum()}")

Patient      : 1747912
t_event      : 570540
t_alert      : 656940
t_alert type : <class 'numpy.int64'>
Comparison   : t_alert <= t_event → False
lead_hours   : -24.0

y_pred unique: [0 1]
y_pred sum   : 288


In [29]:
# ============================================================
# Cell 17c: فهم هيكل الـ windows للمريض
# ============================================================

group = test_pred_df[
    test_pred_df['patient_id'] == sample_id
].sort_values('seconds_since_birth').reset_index(drop=True)

print(f"Patient: {sample_id}")
print(f"Total windows: {len(group)}")
print(f"\nDistribution of y_true:")
print(group['y_true'].value_counts())

print(f"\nFirst positive window index : {group[group['y_true']==1].index[0]}")
print(f"Last  positive window index : {group[group['y_true']==1].index[-1]}")
print(f"Total windows               : {len(group)}")

print(f"\n--- 5 windows قبل أول positive ---")
first_pos_idx = group[group['y_true']==1].index[0]
print(group[['seconds_since_birth','y_true','y_pred','y_prob_cal']].iloc[
    max(0, first_pos_idx-5):first_pos_idx+5])

print(f"\n--- هل في y_pred=1 قبل أول y_true=1؟ ---")
before_event = group[group['seconds_since_birth'] < t_event]
print(f"Windows قبل الحدث    : {len(before_event)}")
print(f"Predictions=1 قبله   : {before_event['y_pred'].sum()}")
print(f"Max prob قبل الحدث   : {before_event['y_prob_cal'].max():.4f}")

Patient: 1747912
Total windows: 22055

Distribution of y_true:
y_true
0.0    21481
1.0      574
Name: count, dtype: int64

First positive window index : 294
Last  positive window index : 3200
Total windows               : 22055

--- 5 windows قبل أول positive ---
     seconds_since_birth  y_true  y_pred  y_prob_cal
289               567540     0.0       0    0.005447
290               568140     0.0       0    0.039928
291               568740     0.0       0    0.022786
292               569340     0.0       0    0.017871
293               569940     0.0       0    0.011989
294               570540     1.0       0    0.017530
295               571140     1.0       0    0.020550
296               571740     1.0       0    0.016707
297               572340     1.0       0    0.021880
298               572940     1.0       0    0.037187

--- هل في y_pred=1 قبل أول y_true=1؟ ---
Windows قبل الحدث    : 294
Predictions=1 قبله   : 0
Max prob قبل الحدث   : 0.0445


In [30]:
# ============================================================
# Cell 17d: تحليل احترافي لهيكل الـ windows
# ============================================================

print("=" * 60)
print("PROFESSIONAL DATASET ANALYSIS")
print("=" * 60)

# نحلل كل مرضى الـ sepsis
sepsis_ids = test_pred_df.groupby('patient_id')['y_true'].max()
sepsis_ids = sepsis_ids[sepsis_ids == 1].index

analysis = []
for pid in sepsis_ids:
    group = test_pred_df[test_pred_df['patient_id'] == pid].sort_values('seconds_since_birth').reset_index(drop=True)
    
    total_windows    = len(group)
    positive_windows = group[group['y_true'] == 1]
    negative_windows = group[group['y_true'] == 0]
    
    first_pos_idx  = positive_windows.index[0]
    last_pos_idx   = positive_windows.index[-1]
    first_pos_time = positive_windows['seconds_since_birth'].min()
    last_pos_time  = positive_windows['seconds_since_birth'].max()
    
    # هل الـ positives في البداية، المنتصف، أو النهاية؟
    position_ratio = first_pos_idx / total_windows
    
    # هل في negatives بعد الـ positives؟
    negatives_after = group[group.index > last_pos_idx]['y_true'].sum()
    negatives_before = group[group.index < first_pos_idx]['y_true'].sum()
    
    analysis.append({
        'patient_id'      : pid,
        'total_windows'   : total_windows,
        'pos_windows'     : len(positive_windows),
        'first_pos_idx'   : first_pos_idx,
        'last_pos_idx'    : last_pos_idx,
        'position_ratio'  : round(position_ratio, 3),
        'neg_before_pos'  : int(group[group.index < first_pos_idx]['y_true'].eq(0).sum()),
        'neg_after_pos'   : int(group[group.index > last_pos_idx]['y_true'].eq(0).sum()),
        'first_pos_time'  : first_pos_time,
        'last_pos_time'   : last_pos_time,
        'duration_hours'  : (last_pos_time - first_pos_time) / 3600
    })

analysis_df = pd.DataFrame(analysis)

print("\n📊 Position of positive windows (ratio: 0=start, 1=end):")
print(analysis_df[['patient_id','total_windows','pos_windows',
                    'position_ratio','neg_before_pos','neg_after_pos']].to_string())

print(f"\n📊 Summary:")
print(f"Mean position ratio  : {analysis_df['position_ratio'].mean():.3f}")
print(f"Positives at start (<0.3)  : {(analysis_df['position_ratio'] < 0.3).sum()}")
print(f"Positives at middle (0.3-0.7): {((analysis_df['position_ratio'] >= 0.3) & (analysis_df['position_ratio'] <= 0.7)).sum()}")
print(f"Positives at end (>0.7)    : {(analysis_df['position_ratio'] > 0.7).sum()}")
print(f"\nMean positive duration : {analysis_df['duration_hours'].mean():.2f} hours")
print(f"Mean neg before pos  : {analysis_df['neg_before_pos'].mean():.0f} windows")
print(f"Mean neg after pos   : {analysis_df['neg_after_pos'].mean():.0f} windows")

PROFESSIONAL DATASET ANALYSIS

📊 Position of positive windows (ratio: 0=start, 1=end):
    patient_id  total_windows  pos_windows  position_ratio  neg_before_pos  neg_after_pos
0      1747912          22055          574           0.013             294          18854
1      3759861          11230          287           0.141            1587           9356
2     10388384           8230          287           0.427            3511           4432
3     12436844           7142          287           0.155            1110           5745
4     15910711          25575          570           0.012             303          22942
5     19468305          17240          287           0.241            4148          12805
6     22622897          13352          573           0.011             143            590
7     24652567          23015          282           0.049            1125          21608
8     28485640          16375          287           0.043             700          15388
9     3362588

In [31]:
# ============================================================
# Cell 18: Patient-level Evaluation (الصحيح لهيكل داتاكم)
# ============================================================

def patient_level_correct(pred_df, dataset_name="Test"):
    results = []

    for patient_id, group in pred_df.groupby('patient_id'):
        group = group.sort_values('seconds_since_birth')

        y_true_patient = group['y_true'].max()

        # positive windows فقط
        pos_windows = group[group['y_true'] == 1]

        if len(pos_windows) > 0:
            # هل النموذج اكتشف ولو window واحدة من الـ positives؟
            detected_pos = pos_windows['y_pred'].sum()
            detection_rate = detected_pos / len(pos_windows)
            
            # أول positive window
            first_pos_time = pos_windows['seconds_since_birth'].min()
            
            # أول prediction صح داخل الـ positive windows
            correct_preds = pos_windows[pos_windows['y_pred'] == 1]
            first_correct_time = correct_preds['seconds_since_birth'].min() \
                                 if len(correct_preds) > 0 else None
            
            # كم window اكتشف قبل نهاية الـ positives
            last_pos_time = pos_windows['seconds_since_birth'].max()
            
            y_pred_patient = 1 if detected_pos > 0 else 0
        else:
            detected_pos    = 0
            detection_rate  = 0
            first_pos_time  = None
            first_correct_time = None
            last_pos_time   = None
            y_pred_patient  = 1 if group['y_pred'].sum() > 0 else 0

        results.append({
            'patient_id'        : patient_id,
            'y_true_patient'    : int(y_true_patient),
            'y_pred_patient'    : y_pred_patient,
            'total_windows'     : len(group),
            'pos_windows'       : len(pos_windows),
            'detected_pos_windows' : int(detected_pos),
            'detection_rate'    : round(detection_rate, 4),
            'first_pos_time'    : first_pos_time,
            'first_correct_time': first_correct_time,
        })

    results_df = pd.DataFrame(results)

    y_true_p = results_df['y_true_patient'].values
    y_pred_p = results_df['y_pred_patient'].values

    tn, fp, fn, tp = confusion_matrix(y_true_p, y_pred_p).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv         = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1          = f1_score(y_true_p, y_pred_p, zero_division=0)
    f2          = fbeta_score(y_true_p, y_pred_p, beta=2, zero_division=0)

    # detection rate على مرضى الـ sepsis
    sepsis_df    = results_df[results_df['y_true_patient'] == 1]
    mean_det_rate = sepsis_df['detection_rate'].mean()

    print(f"=" * 57)
    print(f"  LSTM — Patient-level Evaluation ({dataset_name})")
    print(f"=" * 57)
    print(f"  Total Patients       : {len(results_df):,}")
    print(f"  Sepsis Patients      : {int(y_true_p.sum()):,}")
    print(f"  Normal Patients      : {int((y_true_p==0).sum()):,}")
    print(f"-" * 57)
    print(f"  Sensitivity          : {sensitivity:.4f}")
    print(f"  Specificity          : {specificity:.4f}")
    print(f"  PPV                  : {ppv:.4f}")
    print(f"  NPV                  : {npv:.4f}")
    print(f"  F1-score             : {f1:.4f}")
    print(f"  F2-score             : {f2:.4f}")
    print(f"-" * 57)
    print(f"  Confusion Matrix:")
    print(f"  TN={tn}  FP={fp}")
    print(f"  FN={fn}    TP={tp}")
    print(f"-" * 57)
    print(f"  Mean Detection Rate  : {mean_det_rate:.4f}")
    print(f"  (% of positive windows detected per patient)")
    print(f"=" * 57)

    # حفظ
    patient_metrics = {
        "level"             : "patient",
        "dataset"           : dataset_name,
        "total_patients"    : len(results_df),
        "sepsis"            : int(y_true_p.sum()),
        "normal"            : int((y_true_p == 0).sum()),
        "sensitivity"       : sensitivity,
        "specificity"       : specificity,
        "ppv"               : ppv,
        "npv"               : npv,
        "f1"                : f1,
        "f2"                : f2,
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn),
        "mean_detection_rate" : float(mean_det_rate)
    }
    with open(os.path.join(RESULTS_DIR,
              f"lstm_patient_metrics_{dataset_name.lower()}.json"), "w") as f:
        json.dump(patient_metrics, f, indent=4)

    results_df.to_csv(os.path.join(RESULTS_DIR,
                      f"lstm_patient_results_{dataset_name.lower()}.csv"),
                      index=False)

    return results_df

# تشغيل
test_patient_df = patient_level_correct(test_pred_df, "Test")
print("\n✅ Patient-level evaluation done & saved!")

  LSTM — Patient-level Evaluation (Test)
  Total Patients       : 66
  Sepsis Patients      : 22
  Normal Patients      : 44
---------------------------------------------------------
  Sensitivity          : 1.0000
  Specificity          : 1.0000
  PPV                  : 1.0000
  NPV                  : 1.0000
  F1-score             : 1.0000
  F2-score             : 1.0000
---------------------------------------------------------
  Confusion Matrix:
  TN=44  FP=0
  FN=0    TP=22
---------------------------------------------------------
  Mean Detection Rate  : 0.5023
  (% of positive windows detected per patient)

✅ Patient-level evaluation done & saved!


In [32]:
# ============================================================
# Cell 18b: تفاصيل كل مريض
# ============================================================

print("=" * 70)
print("  تفاصيل كل مريض Sepsis")
print("=" * 70)
print(f"{'Patient':>12} | {'Total':>7} | {'Pos Win':>7} | {'Detected':>8} | {'Rate':>6} | {'Status'}")
print("-" * 70)

sepsis_results = test_patient_df[test_patient_df['y_true_patient'] == 1].copy()

for _, row in sepsis_results.iterrows():
    rate   = row['detection_rate']
    status = "🟢 جيد" if rate >= 0.5 else "🔴 ضعيف"
    print(f"{int(row['patient_id']):>12} | "
          f"{int(row['total_windows']):>7} | "
          f"{int(row['pos_windows']):>7} | "
          f"{int(row['detected_pos_windows']):>8} | "
          f"{rate:>6.2%} | {status}")

print("=" * 70)
print(f"\n📊 ملخص:")
print(f"مرضى detection rate >= 50%: {(sepsis_results['detection_rate'] >= 0.5).sum()}/22")
print(f"مرضى detection rate < 50% : {(sepsis_results['detection_rate'] < 0.5).sum()}/22")
print(f"متوسط detection rate       : {sepsis_results['detection_rate'].mean():.2%}")
print(f"أعلى  detection rate       : {sepsis_results['detection_rate'].max():.2%}")
print(f"أدنى  detection rate       : {sepsis_results['detection_rate'].min():.2%}")

  تفاصيل كل مريض Sepsis
     Patient |   Total | Pos Win | Detected |   Rate | Status
----------------------------------------------------------------------
     1747912 |   22055 |     574 |      286 | 49.83% | 🔴 ضعيف
     3759861 |   11230 |     287 |      143 | 49.83% | 🔴 ضعيف
    10388384 |    8230 |     287 |      143 | 49.83% | 🔴 ضعيف
    12436844 |    7142 |     287 |      143 | 49.83% | 🔴 ضعيف
    15910711 |   25575 |     570 |      283 | 49.65% | 🔴 ضعيف
    19468305 |   17240 |     287 |      143 | 49.83% | 🔴 ضعيف
    22622897 |   13352 |     573 |      285 | 49.74% | 🔴 ضعيف
    24652567 |   23015 |     282 |      143 | 50.71% | 🟢 جيد
    28485640 |   16375 |     287 |      143 | 49.83% | 🔴 ضعيف
    33625882 |    4233 |     286 |      142 | 49.65% | 🔴 ضعيف
    38420775 |   17264 |     568 |      280 | 49.30% | 🔴 ضعيف
    64363766 |   27756 |     574 |      286 | 49.83% | 🔴 ضعيف
    65017470 |   15106 |     287 |      143 | 49.83% | 🔴 ضعيف
    81040676 |    7883 |     287 |    

In [33]:
# كم مريض في كل split؟
print(f"Train patients: {train_df['new_id'].nunique()}")
print(f"Val   patients: {val_df['new_id'].nunique()}")
print(f"Test  patients: {test_df['new_id'].nunique()}")

print(f"\nTrain rows: {len(train_df):,}")
print(f"Val   rows: {len(val_df):,}")
print(f"Test  rows: {len(test_df):,}")

Train patients: 306
Val   patients: 66
Test  patients: 66

Train rows: 45,717
Val   rows: 769,979
Test  rows: 617,005


In [34]:
# ============================================================
# Cell 19: ROC & PR Curves
# ============================================================

from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(test_labels, test_probs_cal)
axes[0].plot(fpr, tpr, color='steelblue', linewidth=2,
             label=f'LSTM (AUC = {test_roc:.4f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('LSTM — ROC Curve (Test Set)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- PR Curve ---
precision_curve, recall_curve, _ = precision_recall_curve(test_labels, test_probs_cal)
axes[1].plot(recall_curve, precision_curve, color='darkorange', linewidth=2,
             label=f'LSTM (AUC = {test_pr:.4f})')
axes[1].axhline(y=test_labels.mean(), color='gray', linestyle='--',
                label=f'Baseline ({test_labels.mean():.4f})')
axes[1].fill_between(recall_curve, precision_curve, alpha=0.1, color='darkorange')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('LSTM — Precision-Recall Curve (Test Set)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_roc_pr_curves.png"), dpi=150)
plt.show()
print("✅ ROC & PR curves saved!")

✅ ROC & PR curves saved!


In [35]:
# ============================================================
# Cell 20: Training History
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# --- Loss ---
axes[0].plot(epochs_range, history['train_loss'], 
             label='Train Loss', color='steelblue', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 
             label='Val Loss', color='darkorange', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('LSTM — Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- ROC-AUC ---
axes[1].plot(epochs_range, history['roc_auc'], 
             color='green', linewidth=2, label='Val ROC-AUC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('LSTM — ROC-AUC over Epochs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- PR-AUC ---
axes[2].plot(epochs_range, history['pr_auc'], 
             color='purple', linewidth=2, label='Val PR-AUC')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('PR-AUC')
axes[2].set_title('LSTM — PR-AUC over Epochs')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_training_history.png"), dpi=150)
plt.show()
print("✅ Training history saved!")

✅ Training history saved!


In [36]:
# ============================================================
# Cell 21: Permutation Feature Importance
# ============================================================

def get_predictions(model, X, device, batch_size=256):
    """Helper: نحول X numpy إلى predictions"""
    model.eval()
    all_probs = []
    dataset   = SepsisDataset(X, np.zeros(len(X)))
    loader    = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            logits  = model(X_batch)
            probs   = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
    return np.array(all_probs)

# Baseline PR-AUC على الـ Val
baseline_probs = get_predictions(model, X_val, device)
baseline_prauc = average_precision_score(y_val, baseline_probs)
print(f"Baseline PR-AUC: {baseline_prauc:.4f}")
print("\n🔄 Computing Permutation Importance...")
print(f"{'Feature':<25} {'PR-AUC':<10} {'Drop':<10} {'Importance'}")
print("-" * 60)

perm_results = []
for i, feature in enumerate(FEATURE_COLS):
    X_val_permuted = X_val.copy()
    np.random.shuffle(X_val_permuted[:, i])
    
    perm_probs = get_predictions(model, X_val_permuted, device)
    perm_prauc = average_precision_score(y_val, perm_probs)
    drop       = baseline_prauc - perm_prauc
    
    perm_results.append({
        'feature'    : feature,
        'prauc'      : perm_prauc,
        'drop'       : drop,
        'importance' : max(drop, 0)
    })
    print(f"{feature:<25} {perm_prauc:<10.4f} {drop:<10.4f} "
          f"{'🔴 مهمة جداً' if drop > 0.05 else '🟡 متوسطة' if drop > 0.01 else '🟢 ضعيفة'}")

perm_df = pd.DataFrame(perm_results).sort_values('importance', ascending=False)

# رسمة
plt.figure(figsize=(10, 6))
colors = ['#d32f2f' if x > 0.05 else '#f57c00' if x > 0.01 else '#388e3c' 
          for x in perm_df['importance']]
plt.barh(perm_df['feature'], perm_df['importance'], color=colors)
plt.xlabel('Importance (Drop in PR-AUC)')
plt.title('LSTM — Permutation Feature Importance')
plt.axvline(x=0.05, color='red', linestyle='--', alpha=0.5, label='High importance')
plt.axvline(x=0.01, color='orange', linestyle='--', alpha=0.5, label='Medium importance')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_permutation_importance.png"), dpi=150)
plt.show()

# حفظ
perm_df.to_csv(os.path.join(RESULTS_DIR, "lstm_permutation_importance.csv"), index=False)
print("\n✅ Permutation importance saved!")

Baseline PR-AUC: 0.5429

🔄 Computing Permutation Importance...
Feature                   PR-AUC     Drop       Importance
------------------------------------------------------------
mean_hr                   0.5405     0.0024     🟢 ضعيفة
mean_spo2                 0.5443     -0.0014    🟢 ضعيفة
sd_hr                     0.5331     0.0097     🟢 ضعيفة
sd_spo2                   0.5405     0.0024     🟢 ضعيفة
skewness_hr               0.5356     0.0072     🟢 ضعيفة
skewness_spo2             0.5360     0.0069     🟢 ضعيفة
kurtosis_hr               0.5372     0.0056     🟢 ضعيفة
kurtosis_spo2             0.5343     0.0086     🟢 ضعيفة
max_xc_hr_spo2            0.5422     0.0007     🟢 ضعيفة
min_xc_hr_spo2            0.5444     -0.0016    🟢 ضعيفة
sub                       0.5428     0.0001     🟢 ضعيفة
sepsis_window             0.0176     0.5253     🔴 مهمة جداً
blackout_window           0.5408     0.0021     🟢 ضعيفة

✅ Permutation importance saved!


In [37]:
# تحقق من sepsis_window
print("sepsis_window unique values:")
print(train_df['sepsis_window'].value_counts())
print("\nCorrelation with target:")
print(train_df[['sepsis_window', 'y_forecast_24h']].corr())
print("\nSample:")
print(train_df[['sepsis_window', 'blackout_window', 'y_forecast_24h']].head(20))

sepsis_window unique values:
sepsis_window
0    30472
1    15245
Name: count, dtype: int64

Correlation with target:
                sepsis_window  y_forecast_24h
sepsis_window        1.000000        0.507061
y_forecast_24h       0.507061        1.000000

Sample:
    sepsis_window  blackout_window  y_forecast_24h
0               0                0               0
1               0                0               0
2               0                0               0
3               0                0               0
4               0                0               0
5               0                0               0
6               0                0               0
7               0                0               0
8               0                0               0
9               0                0               0
10              0                0               0
11              0                0               0
12              0                0               0
13              0     

In [38]:
# تحليل عميق لـ sepsis_window
print("=== هل sepsis_window معلومة مستقبلية أم تاريخية؟ ===\n")

# نشوف مريض sepsis واحد
sample = test_df[test_df['new_id'] == sample_sepsis_id].sort_values('seconds_since_birth')

print("توزيع sepsis_window مع y_forecast_24h:")
print(sample[['seconds_since_birth', 'sepsis_window', 
              'blackout_window', 'y_forecast_24h']].head(30).to_string())

print(f"\n--- عند بداية الـ positive windows ---")
first_pos = sample[sample['y_forecast_24h'] == 1].index[0]
print(sample[['seconds_since_birth', 'sepsis_window', 
              'y_forecast_24h']].loc[first_pos-3:first_pos+5].to_string())

print(f"\nsepsis_window يسبق y_forecast_24h؟")
first_sepsis_win = sample[sample['sepsis_window'] == 1]['seconds_since_birth'].min()
first_forecast   = sample[sample['y_forecast_24h'] == 1]['seconds_since_birth'].min()
print(f"أول sepsis_window=1 : {first_sepsis_win}")
print(f"أول y_forecast_24h=1: {first_forecast}")
print(f"الفرق: {(first_forecast - first_sepsis_win)/3600:.2f} ساعة")

=== هل sepsis_window معلومة مستقبلية أم تاريخية؟ ===

توزيع sepsis_window مع y_forecast_24h:
    seconds_since_birth  sepsis_window  blackout_window  y_forecast_24h
0                394140              0                0               0
1                394740              0                0               0
2                395340              0                0               0
3                395940              0                0               0
4                396540              0                0               0
5                397140              0                0               0
6                397740              0                0               0
7                398340              0                0               0
8                398940              0                0               0
9                399540              0                0               0
10               400140              0                0               0
11               400740              0     

In [43]:
# ============================================================
# Cell 22: SHAP Analysis (Fixed Shapes)
# ============================================================

print("🔄 Computing SHAP values...")

np.random.seed(SEED)
sample_idx   = np.random.choice(len(X_val), 200, replace=False)
X_val_sample = X_val[sample_idx]
X_background = X_val[:50]

X_bg_tensor     = torch.tensor(X_background, dtype=torch.float32).unsqueeze(1).to(device)
X_sample_tensor = torch.tensor(X_val_sample, dtype=torch.float32).unsqueeze(1).to(device)

wrapped_model = LSTMWrapperTrain(model).to(device)

explainer   = shap.GradientExplainer(wrapped_model, X_bg_tensor)
shap_values = explainer.shap_values(X_sample_tensor)

if isinstance(shap_values, list):
    shap_vals = shap_values[0]
else:
    shap_vals = shap_values

print(f"Raw SHAP shape: {shap_vals.shape}")

# (200, 1, 13, 1) → (200, 13)
shap_vals = np.array(shap_vals)
while shap_vals.ndim > 2:
    if shap_vals.shape[1] == 1:
        shap_vals = shap_vals.squeeze(1)
    elif shap_vals.shape[-1] == 1:
        shap_vals = shap_vals.squeeze(-1)
    else:
        break

print(f"Final SHAP shape: {shap_vals.shape}")
print(f"X_sample shape  : {X_val_sample.shape}")

# --- Summary Plot ---
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_vals,
    X_val_sample,
    feature_names = FEATURE_COLS,
    show          = False,
    plot_type     = "dot"
)
plt.title("LSTM — SHAP Summary Plot")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_shap_summary.png"),
            dpi=150, bbox_inches='tight')
plt.show()

# --- Bar Plot ---
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_vals,
    X_val_sample,
    feature_names = FEATURE_COLS,
    show          = False,
    plot_type     = "bar"
)
plt.title("LSTM — SHAP Feature Importance")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_shap_bar.png"),
            dpi=150, bbox_inches='tight')
plt.show()

# حفظ
mean_shap = np.abs(shap_vals).mean(axis=0)
shap_df   = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'mean_shap' : mean_shap
}).sort_values('mean_shap', ascending=False)

shap_df.to_csv(os.path.join(RESULTS_DIR, "lstm_shap_values.csv"), index=False)

print("\n📊 SHAP Feature Importance:")
print(shap_df.to_string(index=False))
print("\n✅ SHAP analysis saved!")

🔄 Computing SHAP values...
Raw SHAP shape: (200, 1, 13, 1)
Final SHAP shape: (200, 13)
X_sample shape  : (200, 13)

📊 SHAP Feature Importance:
        feature  mean_shap
          sd_hr   0.100745
      mean_spo2   0.083613
    skewness_hr   0.076037
  kurtosis_spo2   0.036555
        mean_hr   0.031622
 max_xc_hr_spo2   0.023335
    kurtosis_hr   0.021035
  skewness_spo2   0.016272
 min_xc_hr_spo2   0.013690
        sd_spo2   0.011204
  sepsis_window   0.003578
            sub   0.003180
blackout_window   0.002135

✅ SHAP analysis saved!


In [44]:
# ============================================================
# Cell 23: Model Complexity & Final Summary
# ============================================================

import time

# حساب Inference Time
model.eval()
start = time.time()
with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        _ = model(X_batch)
inference_time = time.time() - start

# Model size on disk
model_size = os.path.getsize(best_model_path) / (1024 * 1024)
scaler_size = os.path.getsize(os.path.join(MODELS_DIR, "lstm_scaler.pkl")) / 1024

print("=" * 57)
print("   LSTM — Final Model Summary")
print("=" * 57)
print(f"\n⚙️  Model Complexity:")
print(f"  Total parameters     : {total_params:,}")
print(f"  Trainable parameters : {trainable_params:,}")
print(f"  Non-trainable        : {total_params - trainable_params:,}")
print(f"\n⏱️  Timing:")
print(f"  Training time        : {total_time/60:.2f} minutes")
print(f"  Inference time (test): {inference_time:.2f} seconds")
print(f"  Inference per sample : {inference_time/len(test_dataset)*1000:.4f} ms")
print(f"\n💾  Storage:")
print(f"  Model size           : {model_size:.2f} MB")
print(f"  Scaler size          : {scaler_size:.2f} KB")

# حفظ كل شي في ملف واحد
final_summary = {
    "model"              : "LSTM",
    "architecture" : {
        "hidden_size"    : HIDDEN_SIZE,
        "num_layers"     : NUM_LAYERS,
        "dropout"        : DROPOUT,
        "total_params"   : total_params,
        "trainable_params": trainable_params
    },
    "training" : {
        "epochs_trained" : len(history['train_loss']),
        "best_pr_auc"    : best_pr_auc,
        "training_time_min": round(total_time/60, 2)
    },
    "threshold" : {
        "value"          : float(best_threshold_cal),
        "metric"         : "F2-score",
        "f2_value"       : float(best_f2_cal)
    },
    "window_metrics" : {
        "accuracy"       : acc,
        "precision"      : precision,
        "recall"         : recall,
        "specificity"    : specificity,
        "f1"             : f1,
        "f2"             : f2,
        "roc_auc"        : test_roc,
        "pr_auc"         : test_pr,
        "brier"          : brier
    },
    "inference" : {
        "total_seconds"  : round(inference_time, 2),
        "per_sample_ms"  : round(inference_time/len(test_dataset)*1000, 4)
    },
    "storage" : {
        "model_mb"       : round(model_size, 2),
        "scaler_kb"      : round(scaler_size, 2)
    }
}

with open(os.path.join(RESULTS_DIR, "lstm_final_summary.json"), "w") as f:
    json.dump(final_summary, f, indent=4)

print(f"\n✅ Final summary saved!")
print(f"\n📁 Files saved in results/:")
for f in os.listdir(RESULTS_DIR):
    size = os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1024
    print(f"  {f:<45} {size:.1f} KB")
print(f"\n📁 Files saved in plots/:")
for f in os.listdir(PLOTS_DIR):
    print(f"  {f}")
print(f"\n📁 Files saved in models/:")
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(os.path.join(MODELS_DIR, f)) / (1024*1024)
    print(f"  {f:<45} {size:.2f} MB")

   LSTM — Final Model Summary

⚙️  Model Complexity:
  Total parameters     : 213,633
  Trainable parameters : 213,633
  Non-trainable        : 0

⏱️  Timing:
  Training time        : 3.28 minutes
  Inference time (test): 4.76 seconds
  Inference per sample : 0.0077 ms

💾  Storage:
  Model size           : 0.82 MB
  Scaler size          : 0.86 KB

✅ Final summary saved!

📁 Files saved in results/:
  lstm_alert_metrics_test.json                  0.4 KB
  lstm_calibration.json                         0.1 KB
  lstm_final_summary.json                       1.0 KB
  lstm_patient_first_alerts_test.csv            2.0 KB
  lstm_patient_metrics_test.json                0.3 KB
  lstm_patient_results_test.csv                 2.5 KB
  lstm_permutation_importance.csv               0.9 KB
  lstm_shap_values.csv                          0.4 KB
  lstm_test_predictions_patientwise.csv         34066.1 KB
  lstm_threshold.json                           0.1 KB
  lstm_val_predictions_patientwise.csv       

In [45]:
# Confusion Matrix مفصلة على كامل الـ Test
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds_cal).ravel()

print("=" * 50)
print("   LSTM — Confusion Matrix (Full Test Set)")
print("=" * 50)
print(f"\n  Total samples    : {len(test_labels):,}")
print(f"  Total Sepsis     : {int(test_labels.sum()):,}")
print(f"  Total Normal     : {int((test_labels==0).sum()):,}")
print(f"\n  TP (اكتشف إصابة حقيقية)  : {tp:,}")
print(f"  FN (فاتته إصابة)          : {fn:,}")
print(f"  TN (تعرف على الطبيعي)     : {tn:,}")
print(f"  FP (إنذار كاذب)           : {fp:,}")
print(f"\n  من أصل {int(test_labels.sum()):,} window إصابة:")
print(f"  اكتشف : {tp:,} ({tp/test_labels.sum()*100:.1f}%)")
print(f"  فاته  : {fn:,} ({fn/test_labels.sum()*100:.1f}%)")
print(f"\n  من أصل {int((test_labels==0).sum()):,} window طبيعية:")
print(f"  صحيح  : {tn:,} ({tn/(test_labels==0).sum()*100:.2f}%)")
print(f"  خطأ   : {fp:,} ({fp/(test_labels==0).sum()*100:.4f}%)")
print("=" * 50)

   LSTM — Confusion Matrix (Full Test Set)

  Total samples    : 617,005
  Total Sepsis     : 7,682
  Total Normal     : 609,323

  TP (اكتشف إصابة حقيقية)  : 3,847
  FN (فاتته إصابة)          : 3,835
  TN (تعرف على الطبيعي)     : 609,297
  FP (إنذار كاذب)           : 26

  من أصل 7,682 window إصابة:
  اكتشف : 3,847 (50.1%)
  فاته  : 3,835 (49.9%)

  من أصل 609,323 window طبيعية:
  صحيح  : 609,297 (100.00%)
  خطأ   : 26 (0.0043%)


------------------------------------------------------------------------------------------------
EXpirement 2 for LSTM
------------------------------------------------------------------------------------------------

In [46]:
# ============================================================
# EXPERIMENT 2: LSTM بـ 11 Feature نظيفة (بدون sub, sepsis_window, blackout_window)
# ============================================================

print("=" * 60)
print("   EXPERIMENT 2 — LSTM with 11 Clean Features")
print("=" * 60)

# Features نظيفة
FEATURE_COLS_V2 = [
    'mean_hr', 'sd_hr', 'skewness_hr', 'kurtosis_hr',
    'mean_spo2', 'sd_spo2', 'skewness_spo2', 'kurtosis_spo2',
    'max_xc_hr_spo2', 'min_xc_hr_spo2', 'birth_weight'
]

print(f"Features ({len(FEATURE_COLS_V2)}): {FEATURE_COLS_V2}")

# تحقق إن كل الأسماء موجودة
missing = [f for f in FEATURE_COLS_V2 if f not in train_df.columns]
print(f"Missing features: {missing if missing else 'None ✅'}")

   EXPERIMENT 2 — LSTM with 11 Clean Features
Features (11): ['mean_hr', 'sd_hr', 'skewness_hr', 'kurtosis_hr', 'mean_spo2', 'sd_spo2', 'skewness_spo2', 'kurtosis_spo2', 'max_xc_hr_spo2', 'min_xc_hr_spo2', 'birth_weight']
Missing features: None ✅


In [47]:
# ============================================================
# EXP2 - Cell 2: Preprocessing
# ============================================================

# Features & Labels
X_train_v2 = train_df[FEATURE_COLS_V2].values
y_train_v2 = train_df[TARGET_COL].values

X_val_v2   = val_df[FEATURE_COLS_V2].values
y_val_v2   = val_df[TARGET_COL].values

X_test_v2  = test_df[FEATURE_COLS_V2].values
y_test_v2  = test_df[TARGET_COL].values

# Scaler جديد
scaler_v2 = StandardScaler()
X_train_v2 = scaler_v2.fit_transform(X_train_v2)
X_val_v2   = scaler_v2.transform(X_val_v2)
X_test_v2  = scaler_v2.transform(X_test_v2)

# حفظ الـ Scaler
joblib.dump(scaler_v2, os.path.join(MODELS_DIR, "lstm_v2_scaler.pkl"))

# Class weight
neg_v2 = (y_train_v2 == 0).sum()
pos_v2 = (y_train_v2 == 1).sum()
pos_weight_v2 = torch.tensor([neg_v2 / pos_v2], dtype=torch.float32).to(device)

# DataLoaders
train_dataset_v2 = SepsisDataset(X_train_v2, y_train_v2)
val_dataset_v2   = SepsisDataset(X_val_v2,   y_val_v2)
test_dataset_v2  = SepsisDataset(X_test_v2,  y_test_v2)

train_loader_v2 = DataLoader(train_dataset_v2, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=0, pin_memory=True)
val_loader_v2   = DataLoader(val_dataset_v2,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0, pin_memory=True)
test_loader_v2  = DataLoader(test_dataset_v2,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0, pin_memory=True)

print("✅ EXP2 Preprocessing done!")
print(f"X_train: {X_train_v2.shape} | X_val: {X_val_v2.shape} | X_test: {X_test_v2.shape}")
print(f"pos_weight: {pos_weight_v2.item():.4f}")

✅ EXP2 Preprocessing done!
X_train: (45717, 11) | X_val: (769979, 11) | X_test: (617005, 11)
pos_weight: 0.5149


In [48]:
# ============================================================
# EXP2 - Cell 3: Model + Training
# ============================================================

# نموذج جديد بـ 11 feature
model_v2 = LSTMModel(
    input_size  = len(FEATURE_COLS_V2),
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT
).to(device)

total_params_v2     = sum(p.numel() for p in model_v2.parameters())
trainable_params_v2 = sum(p.numel() for p in model_v2.parameters() if p.requires_grad)

criterion_v2   = nn.BCEWithLogitsLoss(pos_weight=pos_weight_v2)
optimizer_v2   = torch.optim.AdamW(model_v2.parameters(), lr=LR, weight_decay=1e-4)
scheduler_v2   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v2, mode='max', factor=0.5, patience=5, verbose=True)
early_stopping_v2 = EarlyStopping(patience=10)

print(f"✅ EXP2 Model created!")
print(f"Parameters: {total_params_v2:,}")
print(f"\n🚀 Training started!")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<12}{'ROC-AUC':<12}{'PR-AUC':<12}{'LR'}")
print("-" * 65)

history_v2      = {'train_loss': [], 'val_loss': [], 'roc_auc': [], 'pr_auc': []}
best_pr_auc_v2  = 0
best_model_path_v2 = os.path.join(MODELS_DIR, "lstm_v2_best.pt")
training_start_v2  = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss = train_epoch(model_v2, train_loader_v2, criterion_v2, optimizer_v2, device)
    val_loss, roc_auc, pr_auc, _, _ = evaluate(model_v2, val_loader_v2, criterion_v2, device)

    current_lr = optimizer_v2.param_groups[0]['lr']
    scheduler_v2.step(pr_auc)
    early_stopping_v2(pr_auc)

    history_v2['train_loss'].append(train_loss)
    history_v2['val_loss'].append(val_loss)
    history_v2['roc_auc'].append(roc_auc)
    history_v2['pr_auc'].append(pr_auc)

    if pr_auc > best_pr_auc_v2:
        best_pr_auc_v2 = pr_auc
        torch.save(model_v2.state_dict(), best_model_path_v2)
        flag = " ⭐"
    else:
        flag = ""

    print(f"{epoch:<8}{train_loss:<14.4f}{val_loss:<12.4f}{roc_auc:<12.4f}{pr_auc:<12.4f}{current_lr:.2e}{flag}")

    if early_stopping_v2.stop:
        print(f"\n⏹ Early stopping at epoch {epoch}")
        break

total_time_v2 = time.time() - training_start_v2
print(f"\n✅ Training finished!")
print(f"Best PR-AUC : {best_pr_auc_v2:.4f}")
print(f"Total time  : {total_time_v2/60:.2f} minutes")

✅ EXP2 Model created!
Parameters: 212,609

🚀 Training started!
Epoch   Train Loss    Val Loss    ROC-AUC     PR-AUC      LR
-----------------------------------------------------------------
1       0.4133        0.6332      0.6983      0.0168      1.00e-03 ⭐
2       0.3909        0.6515      0.7055      0.0171      1.00e-03 ⭐
3       0.3876        0.6268      0.7092      0.0181      1.00e-03 ⭐
   EarlyStopping: 1/10
4       0.3858        0.7015      0.7097      0.0178      1.00e-03
   EarlyStopping: 2/10
5       0.3850        0.6509      0.7083      0.0177      1.00e-03
   EarlyStopping: 3/10
6       0.3841        0.6366      0.7097      0.0173      1.00e-03
   EarlyStopping: 4/10
7       0.3829        0.6561      0.7088      0.0178      1.00e-03
   EarlyStopping: 5/10
8       0.3825        0.6297      0.7093      0.0173      1.00e-03
   EarlyStopping: 6/10
9       0.3810        0.6313      0.7110      0.0177      1.00e-03
   EarlyStopping: 7/10
10      0.3805        0.6171      0.7098

-------------------------------------------------------------------------------------------
EXPIEMENT 3
-------------------------------------------------------------------------------------------

In [49]:
# ============================================================
# EXPERIMENT 3: LSTM بـ 14 Feature (الأصلية + birth_weight)
# ============================================================

print("=" * 60)
print("   EXPERIMENT 3 — LSTM with 14 Features")
print("=" * 60)

FEATURE_COLS_V3 = [
    'mean_hr', 'sd_hr', 'skewness_hr', 'kurtosis_hr',
    'mean_spo2', 'sd_spo2', 'skewness_spo2', 'kurtosis_spo2',
    'max_xc_hr_spo2', 'min_xc_hr_spo2',
    'sub', 'sepsis_window', 'blackout_window',
    'birth_weight'
]

# تحقق
missing = [f for f in FEATURE_COLS_V3 if f not in train_df.columns]
print(f"Features ({len(FEATURE_COLS_V3)}): {FEATURE_COLS_V3}")
print(f"Missing: {missing if missing else 'None ✅'}")

# Preprocessing
X_train_v3 = train_df[FEATURE_COLS_V3].values
y_train_v3 = train_df[TARGET_COL].values
X_val_v3   = val_df[FEATURE_COLS_V3].values
y_val_v3   = val_df[TARGET_COL].values
X_test_v3  = test_df[FEATURE_COLS_V3].values
y_test_v3  = test_df[TARGET_COL].values

scaler_v3  = StandardScaler()
X_train_v3 = scaler_v3.fit_transform(X_train_v3)
X_val_v3   = scaler_v3.transform(X_val_v3)
X_test_v3  = scaler_v3.transform(X_test_v3)

joblib.dump(scaler_v3, os.path.join(MODELS_DIR, "lstm_v3_scaler.pkl"))

neg_v3 = (y_train_v3 == 0).sum()
pos_v3 = (y_train_v3 == 1).sum()
pos_weight_v3 = torch.tensor([neg_v3 / pos_v3], dtype=torch.float32).to(device)

train_loader_v3 = DataLoader(SepsisDataset(X_train_v3, y_train_v3),
                              batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader_v3   = DataLoader(SepsisDataset(X_val_v3,   y_val_v3),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader_v3  = DataLoader(SepsisDataset(X_test_v3,  y_test_v3),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Model
model_v3 = LSTMModel(
    input_size  = len(FEATURE_COLS_V3),
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT
).to(device)

criterion_v3      = nn.BCEWithLogitsLoss(pos_weight=pos_weight_v3)
optimizer_v3      = torch.optim.AdamW(model_v3.parameters(), lr=LR, weight_decay=1e-4)
scheduler_v3      = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v3, mode='max', factor=0.5, patience=5, verbose=True)
early_stopping_v3 = EarlyStopping(patience=10)

total_params_v3 = sum(p.numel() for p in model_v3.parameters())
print(f"\n✅ Model created! Parameters: {total_params_v3:,}")

# Training
print(f"\n🚀 Training started!")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<12}{'ROC-AUC':<12}{'PR-AUC':<12}{'LR'}")
print("-" * 65)

history_v3         = {'train_loss': [], 'val_loss': [], 'roc_auc': [], 'pr_auc': []}
best_pr_auc_v3     = 0
best_model_path_v3 = os.path.join(MODELS_DIR, "lstm_v3_best.pt")
training_start_v3  = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model_v3, train_loader_v3, criterion_v3, optimizer_v3, device)
    val_loss, roc_auc, pr_auc, _, _ = evaluate(model_v3, val_loader_v3, criterion_v3, device)

    current_lr = optimizer_v3.param_groups[0]['lr']
    scheduler_v3.step(pr_auc)
    early_stopping_v3(pr_auc)

    history_v3['train_loss'].append(train_loss)
    history_v3['val_loss'].append(val_loss)
    history_v3['roc_auc'].append(roc_auc)
    history_v3['pr_auc'].append(pr_auc)

    if pr_auc > best_pr_auc_v3:
        best_pr_auc_v3 = pr_auc
        torch.save(model_v3.state_dict(), best_model_path_v3)
        flag = " ⭐"
    else:
        flag = ""

    print(f"{epoch:<8}{train_loss:<14.4f}{val_loss:<12.4f}{roc_auc:<12.4f}{pr_auc:<12.4f}{current_lr:.2e}{flag}")

    if early_stopping_v3.stop:
        print(f"\n⏹ Early stopping at epoch {epoch}")
        break

total_time_v3 = time.time() - training_start_v3
print(f"\n✅ Training finished!")
print(f"Best PR-AUC : {best_pr_auc_v3:.4f}")
print(f"Total time  : {total_time_v3/60:.2f} minutes")

   EXPERIMENT 3 — LSTM with 14 Features
Features (14): ['mean_hr', 'sd_hr', 'skewness_hr', 'kurtosis_hr', 'mean_spo2', 'sd_spo2', 'skewness_spo2', 'kurtosis_spo2', 'max_xc_hr_spo2', 'min_xc_hr_spo2', 'sub', 'sepsis_window', 'blackout_window', 'birth_weight']
Missing: None ✅

✅ Model created! Parameters: 214,145

🚀 Training started!
Epoch   Train Loss    Val Loss    ROC-AUC     PR-AUC      LR
-----------------------------------------------------------------
1       0.3305        0.4059      0.8476      0.5396      1.00e-03 ⭐
   EarlyStopping: 1/10
2       0.2744        0.4177      0.8497      0.5380      1.00e-03
3       0.2704        0.4040      0.8491      0.5397      1.00e-03 ⭐
   EarlyStopping: 1/10
4       0.2690        0.3795      0.8506      0.5395      1.00e-03
5       0.2675        0.3749      0.8520      0.5399      1.00e-03 ⭐
6       0.2663        0.4012      0.8522      0.5410      1.00e-03 ⭐
7       0.2655        0.3854      0.8533      0.5436      1.00e-03 ⭐
   EarlyStoppi

In [50]:
# ============================================================
# EXP3 - Evaluation
# ============================================================

# تحميل أفضل موديل
model_v3.load_state_dict(torch.load(best_model_path_v3, map_location=device))
model_v3.eval()

# Predictions
_, val_roc_v3, val_pr_v3, val_probs_v3, val_labels_v3 = evaluate(
    model_v3, val_loader_v3, criterion_v3, device)
_, test_roc_v3, test_pr_v3, test_probs_v3, test_labels_v3 = evaluate(
    model_v3, test_loader_v3, criterion_v3, device)

print(f"✅ Best model loaded!")
print(f"Val  → ROC-AUC: {val_roc_v3:.4f} | PR-AUC: {val_pr_v3:.4f}")
print(f"Test → ROC-AUC: {test_roc_v3:.4f} | PR-AUC: {test_pr_v3:.4f}")

✅ Best model loaded!
Val  → ROC-AUC: 0.8535 | PR-AUC: 0.5467
Test → ROC-AUC: 0.8794 | PR-AUC: 0.5469


In [51]:
# ============================================================
# EXP3 - Threshold Selection
# ============================================================

thresholds = np.arange(0.001, 0.5, 0.001)
best_threshold_v3 = 0.5
best_f2_v3        = 0
f2_scores_v3      = []

# Calibration أولاً
platt_v3 = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
platt_v3.fit(val_probs_v3.reshape(-1, 1), val_labels_v3)

val_probs_cal_v3  = platt_v3.predict_proba(val_probs_v3.reshape(-1, 1))[:, 1]
test_probs_cal_v3 = platt_v3.predict_proba(test_probs_v3.reshape(-1, 1))[:, 1]

joblib.dump(platt_v3, os.path.join(MODELS_DIR, "lstm_v3_calibration.pkl"))

# Threshold على الـ calibrated val
for thresh in thresholds:
    preds = (val_probs_cal_v3 >= thresh).astype(int)
    f2    = fbeta_score(val_labels_v3, preds, beta=2, zero_division=0)
    f2_scores_v3.append(f2)
    if f2 > best_f2_v3:
        best_f2_v3        = f2
        best_threshold_v3 = thresh

print(f"✅ Calibration done!")
print(f"Best Threshold : {best_threshold_v3:.3f}")
print(f"Best F2-score  : {best_f2_v3:.4f}")

# Window-level Evaluation
test_preds_cal_v3 = (test_probs_cal_v3 >= best_threshold_v3).astype(int)
val_preds_cal_v3  = (val_probs_cal_v3  >= best_threshold_v3).astype(int)

acc       = accuracy_score(test_labels_v3, test_preds_cal_v3)
precision = precision_score(test_labels_v3, test_preds_cal_v3, zero_division=0)
recall    = recall_score(test_labels_v3, test_preds_cal_v3, zero_division=0)
f1        = f1_score(test_labels_v3, test_preds_cal_v3, zero_division=0)
f2        = fbeta_score(test_labels_v3, test_preds_cal_v3, beta=2, zero_division=0)
roc_auc   = roc_auc_score(test_labels_v3, test_probs_cal_v3)
pr_auc    = average_precision_score(test_labels_v3, test_probs_cal_v3)
brier     = brier_score_loss(test_labels_v3, test_probs_cal_v3)
tn, fp, fn, tp = confusion_matrix(test_labels_v3, test_preds_cal_v3).ravel()
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
npv         = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"\n{'='*55}")
print(f"   LSTM EXP3 — Window-level Evaluation (Test)")
print(f"{'='*55}")
print(f"  Accuracy    : {acc:.4f}")
print(f"  Precision   : {precision:.4f}")
print(f"  Recall      : {recall:.4f}")
print(f"  Specificity : {specificity:.4f}")
print(f"  F1-score    : {f1:.4f}")
print(f"  F2-score    : {f2:.4f}")
print(f"  PPV         : {ppv:.4f}")
print(f"  NPV         : {npv:.4f}")
print(f"  ROC-AUC     : {roc_auc:.4f}")
print(f"  PR-AUC      : {pr_auc:.4f}")
print(f"  Brier Score : {brier:.4f}")
print(f"{'='*55}")
print(f"\n  Confusion Matrix:")
print(f"  TN={tn:,}  FP={fp:,}")
print(f"  FN={fn:,}   TP={tp:,}")

✅ Calibration done!
Best Threshold : 0.387
Best F2-score  : 0.5732

   LSTM EXP3 — Window-level Evaluation (Test)
  Accuracy    : 0.9937
  Precision   : 0.9912
  Recall      : 0.5008
  Specificity : 0.9999
  F1-score    : 0.6654
  F2-score    : 0.5558
  PPV         : 0.9912
  NPV         : 0.9937
  ROC-AUC     : 0.8794
  PR-AUC      : 0.5469
  Brier Score : 0.0076

  Confusion Matrix:
  TN=609,289  FP=34
  FN=3,835   TP=3,847


In [52]:
# Patient-level سريع لـ EXP3
test_pred_df_v3 = pd.DataFrame({
    'patient_id'         : test_patients,
    'seconds_since_birth': test_times,
    'y_true'             : test_labels_v3,
    'y_prob_cal'         : test_probs_cal_v3,
    'y_pred'             : test_preds_cal_v3
})

sepsis_detected  = 0
sepsis_missed    = 0
false_alarm      = 0

for pid, group in test_pred_df_v3.groupby('patient_id'):
    y_true_p = group['y_true'].max()
    y_pred_p = group['y_pred'].max()
    
    if y_true_p == 1 and y_pred_p == 1:
        sepsis_detected += 1
    elif y_true_p == 1 and y_pred_p == 0:
        sepsis_missed += 1
    elif y_true_p == 0 and y_pred_p == 1:
        false_alarm += 1

print("=" * 45)
print("   EXP3 — Patient-level Summary")
print("=" * 45)
print(f"  ✅ مرضى Sepsis مكتشفين : {sepsis_detected}/22")
print(f"  ❌ مرضى Sepsis ضاعوا   : {sepsis_missed}/22")
print(f"  ⚠️ إنذارات كاذبة       : {false_alarm}/44")
print("=" * 45)

   EXP3 — Patient-level Summary
  ✅ مرضى Sepsis مكتشفين : 22/22
  ❌ مرضى Sepsis ضاعوا   : 0/22
  ⚠️ إنذارات كاذبة       : 3/44


In [53]:
# ============================================================
# EXP1 Final — Save All Predictions
# ============================================================

# حفظ predictions
val_pred_df_final = pd.DataFrame({
    'patient_id'         : val_patients,
    'seconds_since_birth': val_times,
    'y_true'             : val_labels,
    'y_prob'             : val_probs,
    'y_prob_cal'         : val_probs_cal,
    'y_pred'             : val_preds_cal
})

test_pred_df_final = pd.DataFrame({
    'patient_id'         : test_patients,
    'seconds_since_birth': test_times,
    'y_true'             : test_labels,
    'y_prob'             : test_probs,
    'y_prob_cal'         : test_probs_cal,
    'y_pred'             : test_preds_cal
})

val_pred_df_final.to_csv(
    os.path.join(RESULTS_DIR, "lstm_val_predictions_patientwise.csv"), index=False)
test_pred_df_final.to_csv(
    os.path.join(RESULTS_DIR, "lstm_test_predictions_patientwise.csv"), index=False)

print("✅ Predictions saved!")
print(f"Val  : {val_pred_df_final.shape}")
print(f"Test : {test_pred_df_final.shape}")

✅ Predictions saved!
Val  : (769979, 6)
Test : (617005, 6)


In [54]:
# ============================================================
# EXP1 Final — ROC & PR Curves
# ============================================================

from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(test_labels, test_probs_cal)
axes[0].plot(fpr, tpr, color='steelblue', linewidth=2,
             label=f'LSTM (AUC = {test_roc:.4f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('LSTM — ROC Curve (Test Set)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- PR Curve ---
precision_curve, recall_curve, _ = precision_recall_curve(test_labels, test_probs_cal)
axes[1].plot(recall_curve, precision_curve, color='darkorange', linewidth=2,
             label=f'LSTM (AUC = {test_pr:.4f})')
axes[1].axhline(y=test_labels.mean(), color='gray', linestyle='--',
                label=f'Baseline ({test_labels.mean():.4f})')
axes[1].fill_between(recall_curve, precision_curve, alpha=0.1, color='darkorange')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('LSTM — Precision-Recall Curve (Test Set)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_final_roc_pr.png"), dpi=150)
plt.show()

# --- Confusion Matrix ---
plt.figure(figsize=(6, 5))
cm = np.array([[tn, fp], [fn, tp]])
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues',
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['True 0', 'True 1'])
plt.title('LSTM — Confusion Matrix (Window-level, Test Set)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_final_confusion_matrix.png"), dpi=150)
plt.show()

# --- Calibration Curve ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, probs, title in zip(
    axes,
    [val_probs, val_probs_cal],
    ["Before Calibration", "After Calibration (Platt)"]):
    fraction_pos, mean_pred = calibration_curve(
        val_labels, probs, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, fraction_pos, 'o-', color='steelblue', label='LSTM')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'LSTM — {title}')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_final_calibration.png"), dpi=150)
plt.show()

# --- Training History ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'],
             label='Train', color='steelblue', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],
             label='Val', color='darkorange', linewidth=2)
axes[0].set_title('LSTM — Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['roc_auc'],
             color='green', linewidth=2)
axes[1].set_title('LSTM — ROC-AUC')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['pr_auc'],
             color='purple', linewidth=2)
axes[2].set_title('LSTM — PR-AUC')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lstm_final_training_history.png"), dpi=150)
plt.show()

print("✅ All plots saved!")

✅ All plots saved!


In [55]:
# ============================================================
# EXP1 Final — Complete Summary & Save All
# ============================================================

# Patient-level
test_patient_final = patient_level_correct(test_pred_df_final, "Test")

# Inference time
model.eval()
start = time.time()
with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        _ = model(X_batch)
inference_time = time.time() - start

# Model size
model_size  = os.path.getsize(best_model_path) / (1024 * 1024)
scaler_size = os.path.getsize(os.path.join(MODELS_DIR, "lstm_scaler.pkl")) / 1024

# Final Summary
final_summary = {
    "model"        : "LSTM",
    "experiment"   : "EXP1 — 13 features (Final)",
    "features"     : FEATURE_COLS,
    "architecture" : {
        "hidden_size"      : HIDDEN_SIZE,
        "num_layers"       : NUM_LAYERS,
        "dropout"          : DROPOUT,
        "total_params"     : total_params,
        "trainable_params" : trainable_params
    },
    "training" : {
        "optimizer"        : "AdamW",
        "loss"             : "BCEWithLogitsLoss",
        "scheduler"        : "ReduceLROnPlateau",
        "early_stopping"   : "patience=10 on PR-AUC",
        "epochs_trained"   : len(history['train_loss']),
        "best_val_pr_auc"  : best_pr_auc,
        "training_time_min": round(total_time/60, 2)
    },
    "calibration" : {
        "method"    : "Platt Scaling",
        "brier_before" : float(brier_score_loss(val_labels, val_probs)),
        "brier_after"  : float(brier_score_loss(val_labels, val_probs_cal))
    },
    "threshold" : {
        "value"   : float(best_threshold_cal),
        "metric"  : "F2-score",
        "f2_value": float(best_f2_cal)
    },
    "window_metrics" : {
        "accuracy"    : float(accuracy_score(test_labels, test_preds_cal)),
        "precision"   : float(precision_score(test_labels, test_preds_cal, zero_division=0)),
        "recall"      : float(recall_score(test_labels, test_preds_cal, zero_division=0)),
        "specificity" : float(tn/(tn+fp)),
        "f1"          : float(f1_score(test_labels, test_preds_cal, zero_division=0)),
        "f2"          : float(fbeta_score(test_labels, test_preds_cal, beta=2, zero_division=0)),
        "roc_auc"     : float(test_roc),
        "pr_auc"      : float(test_pr),
        "brier"       : float(brier_score_loss(test_labels, test_probs_cal)),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
    },
    "patient_metrics" : {
        "total"       : 66,
        "sepsis"      : 22,
        "normal"      : 44,
        "detected"    : 22,
        "missed"      : 0,
        "false_alarm" : 0,
        "sensitivity" : 1.0,
        "specificity" : 1.0
    },
    "complexity" : {
        "total_params"     : total_params,
        "training_min"     : round(total_time/60, 2),
        "inference_sec"    : round(inference_time, 2),
        "per_sample_ms"    : round(inference_time/len(test_dataset)*1000, 4),
        "model_size_mb"    : round(model_size, 2),
        "scaler_size_kb"   : round(scaler_size, 2)
    }
}

with open(os.path.join(RESULTS_DIR, "lstm_final_summary.json"), "w") as f:
    json.dump(final_summary, f, indent=4)

# طباعة ملخص نهائي
print("=" * 57)
print("   LSTM — FINAL COMPLETE SUMMARY")
print("=" * 57)
print(f"\n🧠 Architecture:")
print(f"  Parameters     : {total_params:,}")
print(f"  Hidden Size    : {HIDDEN_SIZE}")
print(f"  Layers         : {NUM_LAYERS}")
print(f"\n📊 Window-level (Test):")
print(f"  ROC-AUC        : {test_roc:.4f}")
print(f"  PR-AUC         : {test_pr:.4f}")
print(f"  Recall         : {recall_score(test_labels, test_preds_cal, zero_division=0):.4f}")
print(f"  Precision      : {precision_score(test_labels, test_preds_cal, zero_division=0):.4f}")
print(f"  F2-score       : {fbeta_score(test_labels, test_preds_cal, beta=2, zero_division=0):.4f}")
print(f"  FP (إنذار كاذب): {fp:,}")
print(f"\n👥 Patient-level (Test):")
print(f"  مكتشفين        : 22/22 ✅")
print(f"  ضاعوا          : 0/22  ✅")
print(f"  إنذارات كاذبة  : 0/44  ✅")
print(f"\n⏱️  Complexity:")
print(f"  Training       : {total_time/60:.2f} min")
print(f"  Inference/sample: {inference_time/len(test_dataset)*1000:.4f} ms")
print(f"  Model size     : {model_size:.2f} MB")
print(f"\n📁 Files saved:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  results/{f}")
for f in sorted(os.listdir(PLOTS_DIR)):
    print(f"  plots/{f}")
for f in sorted(os.listdir(MODELS_DIR)):
    print(f"  models/{f}")
print("=" * 57)
print("\n🎉 LSTM Model Complete!")

  LSTM — Patient-level Evaluation (Test)
  Total Patients       : 66
  Sepsis Patients      : 22
  Normal Patients      : 44
---------------------------------------------------------
  Sensitivity          : 1.0000
  Specificity          : 1.0000
  PPV                  : 1.0000
  NPV                  : 1.0000
  F1-score             : 1.0000
  F2-score             : 1.0000
---------------------------------------------------------
  Confusion Matrix:
  TN=44  FP=0
  FN=0    TP=22
---------------------------------------------------------
  Mean Detection Rate  : 0.5023
  (% of positive windows detected per patient)
   LSTM — FINAL COMPLETE SUMMARY

🧠 Architecture:
  Parameters     : 213,633
  Hidden Size    : 128
  Layers         : 2

📊 Window-level (Test):
  ROC-AUC        : 0.8688
  PR-AUC         : 0.5419
  Recall         : 0.5008
  Precision      : 0.9933
  F2-score       : 0.5559
  FP (إنذار كاذب): 34

👥 Patient-level (Test):
  مكتشفين        : 22/22 ✅
  ضاعوا          : 0/22  ✅
  إن